# KDB Family — Multi-Dataset Comparative Benchmark

**Frozen implementation provenance**

- Source notebook: `KDB Family - Complete Skin and MiniBooNE Benchmark.ipynb`
- Repository commit: `cf55d7c5570f8304c618c9883dd9c8e364a85a3d`
- Source notebook SHA-256: `2BF6A258EDC878E6A7671644B5539AC38BD5BF802800982102CD8F80FBD60232`
- Generated benchmark file: `KDB Family - Multi-Dataset Comparative Benchmark.ipynb`

The learner and MDL implementation cells below are copied from the source
notebook without algorithmic changes. The source notebook is the frozen
reference and is never modified by this notebook.


## 1. Purpose and methodological contract

This is the single canonical, top-to-bottom executable analysis for a
multi-dataset KDB-family comparison. It evaluates KDB, published sKDB,
dKDB, and **experimental/unpublished sdKDB** using identical outer splits
and one shared training-fold-fitted MDL discretization per split.

The headline comparison is deliberately capacity matched:

| Learner | Headline configuration | Learner passes |
|---|---|---:|
| KDB | `k=5` | 2 |
| sKDB | `kmax=5` | 3 |
| dKDB | `k=5, D=1` | 3 |
| sdKDB **(experimental/unpublished)** | `kmax=5, D=1` | 4 |

`k=0…5` is a separate sensitivity experiment; it is not substituted for
this matched headline comparison. No hyperparameter tuning is hidden from
pass accounting.

The supervised Fayyad–Irani-style MDL preprocessor is numeric-only and
in-memory. Preprocessing time and rows are reported separately from
learner passes, so this experiment measures limited-pass learner behavior
separately from preprocessing. For every outer split a fresh MDL object is
fitted on training rows only, then transforms train/test separately; all
four learners reuse that same discretized fold.

True-class RMSE is `sqrt(mean((1 - p_true)^2))`. Multiclass Brier score is
`mean_i sum_c (p_ic - 1[y_i=c])^2` (no division by class count). External
stratified CV is distinct from sKDB's internal third-pass incremental
LOOCV selection.


## 2. Imports and centralized configuration


In [1]:
from __future__ import annotations

import gc
import gzip
import json
import math
import os
import pickle
import platform
import sys
import urllib.request
import warnings
import zipfile
from collections.abc import Callable, Iterator, Mapping, Sequence
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from hashlib import sha256
from math import log2
from pathlib import Path
from time import perf_counter

import certifi
import ssl
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare, rankdata, studentized_range, t, wilcoxon
from sklearn.datasets import fetch_covtype, fetch_openml, load_svmlight_file
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.22})

SOURCE_NOTEBOOK = 'KDB Family - Complete Skin and MiniBooNE Benchmark.ipynb'
SOURCE_NOTEBOOK_SHA256 = '2BF6A258EDC878E6A7671644B5539AC38BD5BF802800982102CD8F80FBD60232'
REPOSITORY_COMMIT = 'cf55d7c5570f8304c618c9883dd9c8e364a85a3d'

# ------------------------- ONE CONFIGURATION CELL -------------------------
MODE = "CORE"  # "SMOKE" | "CORE" | "STRESS" | "PAPER_FIDELITY"

RANDOM_SEED = 5
K = 5
KMAX = 5
D = 1
OPTIMIZER = "adagrad"
LEARNING_RATE = 0.01
ADAGRAD_EPSILON = 1e-9
SENSITIVITY_K_VALUES = tuple(range(6))
STRESS_SCALING_ROWS = (100_000, 250_000, 500_000, 1_000_000, 2_000_000, None)

MODE_CONFIGS = {
    "SMOKE": {
        "datasets": ("Skin Segmentation", "MiniBooNE"),
        "folds": 2, "repeats": 2,
        "row_limits": {"Skin Segmentation": 300, "MiniBooNE": 200},
        "use_predefined_holdout": False,
    },
    "CORE": {
        "datasets": ("Skin Segmentation", "MiniBooNE", "cod-rna", "Covertype", "YearPredictionMSD"),
        "folds": 3, "repeats": 1, "row_limits": {},
        "use_predefined_holdout": False,
    },
    "STRESS": {
        "datasets": ("SUSY", "HIGGS"),
        "folds": 1, "repeats": 1, "row_limits": {},
        "use_predefined_holdout": True,
    },
    "PAPER_FIDELITY": {
        "datasets": ("Skin Segmentation", "MiniBooNE", "cod-rna", "Covertype", "YearPredictionMSD"),
        "folds": 10, "repeats": 1, "row_limits": {},
        "use_predefined_holdout": False,
    },
}

RUN_BENCHMARKS = True
RUN_K_SENSITIVITY = True
ENABLE_RESULT_CACHE = True
RESUME_COMPLETED_RUNS = True
CACHE_SCHEMA_VERSION = 2
DATA_CACHE_DIR = Path("KDB family benchmark cache") / "datasets"
RESULT_CACHE_ROOT = Path("KDB family benchmark cache") / "results"

if MODE not in MODE_CONFIGS:
    raise ValueError(f"Unknown MODE={MODE!r}")
if K != 5 or KMAX != 5 or K != KMAX:
    raise AssertionError("Headline research comparison must remain matched at k=5/kmax=5")
if D != 1:
    raise AssertionError("Headline research comparison requires D=1")
ACTIVE_MODE = MODE_CONFIGS[MODE]

HEADLINE_MODEL_SPECS = {
    "KDB": {"k": K},
    "sKDB": {"kmax": KMAX},
    "dKDB": {"k": K, "D": D, "optimizer": OPTIMIZER, "learning_rate": LEARNING_RATE},
    "sdKDB (experimental/unpublished)": {"kmax": KMAX, "D": D, "optimizer": OPTIMIZER, "learning_rate": LEARNING_RATE},
}

CONFIG_PAYLOAD = {
    "schema": CACHE_SCHEMA_VERSION, "mode": MODE, "mode_config": ACTIVE_MODE,
    "seed": RANDOM_SEED, "k": K, "kmax": KMAX, "D": D,
    "optimizer": OPTIMIZER, "learning_rate": LEARNING_RATE,
    "adagrad_epsilon": ADAGRAD_EPSILON,
    "source_sha256": SOURCE_NOTEBOOK_SHA256,
}
CONFIGURATION_ID = sha256(
    json.dumps(CONFIG_PAYLOAD, sort_keys=True, default=str).encode("utf-8")
).hexdigest()[:16]

display(pd.DataFrame.from_dict(HEADLINE_MODEL_SPECS, orient="index").fillna("—"))
print(f"MODE={MODE} | configuration_id={CONFIGURATION_ID} | datasets={ACTIVE_MODE['datasets']}")


,k,kmax,D,optimizer,learning_rate
KDB,5.0,—,—,—,—
dKDB,5.0,—,1.0,adagrad,0.01
sKDB,—,5.0,—,—,—
sdKDB (experimental/unpublished),—,5.0,1.0,adagrad,0.01


MODE=CORE | configuration_id=53e7c82e27eb953c | datasets=('Skin Segmentation', 'MiniBooNE', 'cod-rna', 'Covertype', 'YearPredictionMSD')


## 3. Frozen learner implementation


### Shared in-memory MDL preprocessing

Copied from the frozen source. Numeric input is mandatory; arbitrary
categorical variables are not ordinal-encoded. Nonfinite numeric values
use a reserved state.


In [2]:
def _entropy(counts: np.ndarray) -> float:
    total = int(counts.sum())
    if total == 0:
        return 0.0
    probabilities = counts[counts > 0] / total
    return float(-np.sum(probabilities * np.log2(probabilities)))


def _mdl_cut_points(values: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Return recursively accepted Fayyad-Irani MDL cuts for one feature."""

    if values.size < 2 or np.unique(labels).size < 2:
        return np.empty(0, dtype=np.float64)

    order = np.argsort(values, kind="mergesort")
    sorted_values = values[order]
    sorted_labels = labels[order]
    n_classes = int(sorted_labels.max()) + 1
    accepted: list[float] = []

    def split(start: int, stop: int) -> None:
        n = stop - start
        if n < 2:
            return

        node_labels = sorted_labels[start:stop]
        total_counts = np.bincount(node_labels, minlength=n_classes)
        parent_entropy = _entropy(total_counts)
        if parent_entropy == 0.0:
            return

        left_counts = np.zeros(n_classes, dtype=np.int64)
        best_position: int | None = None
        best_weighted_entropy = np.inf
        for position in range(start, stop - 1):
            left_counts[sorted_labels[position]] += 1
            if sorted_values[position] == sorted_values[position + 1]:
                continue
            n_left = position - start + 1
            n_right = n - n_left
            right_counts = total_counts - left_counts
            weighted = (
                n_left * _entropy(left_counts)
                + n_right * _entropy(right_counts)
            ) / n
            if weighted < best_weighted_entropy:
                best_weighted_entropy = weighted
                best_position = position

        if best_position is None:
            return

        left = np.bincount(
            sorted_labels[start : best_position + 1], minlength=n_classes
        )
        right = total_counts - left
        gain = parent_entropy - best_weighted_entropy
        k = int(np.count_nonzero(total_counts))
        k_left = int(np.count_nonzero(left))
        k_right = int(np.count_nonzero(right))
        delta = log2((3**k) - 2) - (
            k * parent_entropy
            - k_left * _entropy(left)
            - k_right * _entropy(right)
        )
        threshold = (log2(n - 1) + delta) / n
        if gain <= threshold:
            return

        low = float(sorted_values[best_position])
        high = float(sorted_values[best_position + 1])
        accepted.append(low + (high - low) / 2.0)
        split(start, best_position + 1)
        split(best_position + 1, stop)

    split(0, sorted_values.size)
    return np.asarray(sorted(accepted), dtype=np.float64)


class MDLDiscretizer:
    """Training-fold-only, in-memory supervised MDL discretizer.

    A permanently reserved state handles NaN and +/- infinity.  The public fit
    audit fields are used by the evaluator and leakage tests.
    """

    method_name = "Fayyad-Irani supervised MDL (in memory; training fold only)"

    def __init__(self) -> None:
        self.cut_points_: list[np.ndarray] | None = None
        self.cardinalities_: np.ndarray | None = None
        self.fit_seconds_: float = 0.0
        self.fit_rows_: int = 0
        self.fit_calls_: int = 0

    def fit(self, X: Sequence, y: Sequence) -> "MDLDiscretizer":
        started = perf_counter()
        X_array = _numeric_matrix(X)
        y_array = np.asarray(y, dtype=object).reshape(-1)
        if len(X_array) != len(y_array):
            raise ValueError("X and y must contain the same number of rows")
        if len(X_array) == 0:
            raise ValueError("Cannot fit MDLDiscretizer on an empty fold")

        label_to_index: dict[object, int] = {}
        encoded_y = np.empty(len(y_array), dtype=np.int32)
        for row, label in enumerate(y_array.tolist()):
            try:
                encoded_y[row] = label_to_index[label]
            except KeyError:
                label_to_index[label] = len(label_to_index)
                encoded_y[row] = label_to_index[label]

        self.cut_points_ = []
        for feature in range(X_array.shape[1]):
            observed = np.isfinite(X_array[:, feature])
            self.cut_points_.append(
                _mdl_cut_points(
                    X_array[observed, feature], encoded_y[observed]
                )
            )

        # digitize yields len(cuts)+1 observed bins; reserve one more for missing.
        self.cardinalities_ = np.asarray(
            [len(cuts) + 2 for cuts in self.cut_points_], dtype=np.int32
        )
        self.fit_rows_ = int(len(X_array))
        self.fit_calls_ += 1
        self.fit_seconds_ = float(perf_counter() - started)
        return self

    def transform(self, X: Sequence) -> np.ndarray:
        if self.cut_points_ is None or self.cardinalities_ is None:
            raise RuntimeError("MDLDiscretizer must be fitted before transform")
        X_array = _numeric_matrix(X)
        if X_array.shape[1] != len(self.cut_points_):
            raise ValueError("Transform data has a different feature count")

        transformed = np.empty(X_array.shape, dtype=np.int32)
        for feature, cuts in enumerate(self.cut_points_):
            observed = np.isfinite(X_array[:, feature])
            transformed[observed, feature] = np.digitize(
                X_array[observed, feature], cuts, right=False
            )
            transformed[~observed, feature] = len(cuts) + 1
        return transformed

    def fit_transform(self, X: Sequence, y: Sequence) -> np.ndarray:
        return self.fit(X, y).transform(X)


def _numeric_matrix(X: Sequence) -> np.ndarray:
    if hasattr(X, "to_numpy"):
        X = X.to_numpy(copy=False)
    array = np.asarray(X)
    if array.ndim != 2:
        raise ValueError("X must be a two-dimensional matrix")
    try:
        return array.astype(np.float64, copy=False)
    except (TypeError, ValueError) as exc:
        raise TypeError("MDLDiscretizer requires numeric features") from exc


### Auditable learner passes


In [3]:
@dataclass(frozen=True)
class PassRecord:
    stage: str
    pass_number: int
    rows_processed: int
    seconds: float


class PassTracker:
    """Records only full learner traversals (preprocessing is separate)."""

    def __init__(self, expected_rows: int) -> None:
        if expected_rows <= 0:
            raise ValueError("expected_rows must be positive")
        self.expected_rows = int(expected_rows)
        self.records: list[PassRecord] = []

    @contextmanager
    def iter_rows(
        self, stage: str, X: np.ndarray, y: np.ndarray
    ) -> Iterator[Iterator[tuple[np.ndarray, int]]]:
        processed = 0
        started = perf_counter()

        def rows() -> Iterator[tuple[np.ndarray, int]]:
            nonlocal processed
            for index in range(self.expected_rows):
                processed += 1
                yield X[index], int(y[index])

        yield rows()
        if processed != self.expected_rows:
            raise RuntimeError(
                f"{stage} processed {processed} rows; expected "
                f"{self.expected_rows}"
            )
        self.records.append(
            PassRecord(
                stage=stage,
                pass_number=len(self.records) + 1,
                rows_processed=processed,
                seconds=float(perf_counter() - started),
            )
        )

    @contextmanager
    def iter_batches(
        self,
        stage: str,
        X: np.ndarray,
        y: np.ndarray,
        batch_size: int,
    ) -> Iterator[Iterator[tuple[np.ndarray, np.ndarray]]]:
        processed = 0
        started = perf_counter()

        def batches() -> Iterator[tuple[np.ndarray, np.ndarray]]:
            nonlocal processed
            for start in range(0, self.expected_rows, batch_size):
                stop = min(start + batch_size, self.expected_rows)
                processed += stop - start
                yield X[start:stop], y[start:stop]

        yield batches()
        if processed != self.expected_rows:
            raise RuntimeError(
                f"{stage} processed {processed} rows; expected "
                f"{self.expected_rows}"
            )
        self.records.append(
            PassRecord(
                stage=stage,
                pass_number=len(self.records) + 1,
                rows_processed=processed,
                seconds=float(perf_counter() - started),
            )
        )

    @property
    def pass_count(self) -> int:
        return len(self.records)

    @property
    def rows_processed(self) -> int:
        return int(sum(record.rows_processed for record in self.records))

    def as_records(self) -> list[dict[str, object]]:
        return [asdict(record) for record in self.records]


### KDB (including the true one-pass `k=0` path)


In [4]:
class KDBClassifier:
    """K-dependence Bayesian classifier for discrete, non-negative features.

    ``k=0`` is a genuine one-pass Naive Bayes path: it skips MI, CMI and all
    pair-count allocation.  ``k>0`` uses one structure pass and one parameter
    pass, as in Fupla's KDB stages.
    """

    model_name = "KDB"

    def __init__(self, k: int = 1, *, m_estimate: float = 1.0) -> None:
        if int(k) != k or k < 0:
            raise ValueError("k must be a non-negative integer")
        if m_estimate <= 0:
            raise ValueError("m_estimate must be positive")
        self.k = int(k)
        self.m_estimate = float(m_estimate)

    def fit(self, X: Sequence, y: Sequence) -> "KDBClassifier":
        X_array = _categorical_matrix(X)
        y_encoded, classes, label_to_index = _encode_labels(y)
        if len(X_array) != len(y_encoded):
            raise ValueError("X and y must contain the same number of rows")
        if len(X_array) == 0 or X_array.shape[1] == 0:
            raise ValueError("KDB requires at least one row and one feature")

        self.classes_ = classes
        self._label_to_index_ = label_to_index
        self.n_classes_ = len(classes)
        self.n_features_in_ = X_array.shape[1]
        self.n_train_rows_ = len(X_array)
        self.cardinalities_ = X_array.max(axis=0).astype(np.int64) + 1
        self.pass_tracker_ = PassTracker(len(X_array))
        self.structure_learned_ = self.k > 0
        self.selection_rmse_ = None
        self.selected_k_ = min(self.k, self.n_features_in_ - 1)
        self.selected_feature_count_ = self.n_features_in_
        self.discriminative_iterations_ = 0
        self.discriminative_updates_ = 0

        if self.k == 0:
            self.feature_order_ = np.arange(self.n_features_in_, dtype=np.int64)
            self.parents_ = [np.empty(0, dtype=np.int64) for _ in self.feature_order_]
            self.class_counts_ = np.zeros(self.n_classes_, dtype=np.int64)
            self.tables_ = [
                [
                    {
                        (): np.zeros(
                            (int(self.cardinalities_[feature]), self.n_classes_),
                            dtype=np.int64,
                        )
                    }
                ]
                for feature in self.feature_order_
            ]
            with self.pass_tracker_.iter_rows(
                "naive_bayes_parameters", X_array, y_encoded
            ) as rows:
                for row, target in rows:
                    self.class_counts_[target] += 1
                    for position, feature in enumerate(self.feature_order_):
                        self.tables_[position][0][()][int(row[feature]), target] += 1
        else:
            self._fit_structure(X_array, y_encoded)
            self._fit_parameters(X_array, y_encoded)

        self.full_feature_order_ = self.feature_order_.copy()
        self.training_seconds_ = float(
            sum(record.seconds for record in self.pass_tracker_.records)
        )
        return self

    def _fit_structure(self, X: np.ndarray, y: np.ndarray) -> None:
        n_features = self.n_features_in_
        n_classes = self.n_classes_
        xy_offsets = np.zeros(n_features, dtype=np.int64)
        if n_features > 1:
            xy_offsets[1:] = np.cumsum(self.cardinalities_[:-1])
        xy_counts = np.zeros(
            (int(self.cardinalities_.sum()), n_classes), dtype=np.int64
        )
        class_counts = np.zeros(n_classes, dtype=np.int64)

        pair_i: list[int] = []
        pair_j: list[int] = []
        pair_offsets: list[int] = []
        pair_state_total = 0
        for i in range(1, n_features):
            for j in range(i):
                pair_i.append(i)
                pair_j.append(j)
                pair_offsets.append(pair_state_total)
                pair_state_total += int(self.cardinalities_[i] * self.cardinalities_[j])
        pair_i_array = np.asarray(pair_i, dtype=np.int64)
        pair_j_array = np.asarray(pair_j, dtype=np.int64)
        pair_offsets_array = np.asarray(pair_offsets, dtype=np.int64)
        pair_counts = np.zeros((pair_state_total, n_classes), dtype=np.int64)

        n_pairs = max(1, len(pair_i))
        batch_size = max(1, min(512, 1_000_000 // n_pairs))
        with self.pass_tracker_.iter_batches(
            "kdb_structure_mi_cmi", X, y, batch_size
        ) as batches:
            for X_batch, y_batch in batches:
                np.add.at(class_counts, y_batch, 1)
                xy_states = xy_offsets[None, :] + X_batch
                for class_index in range(n_classes):
                    mask = y_batch == class_index
                    if np.any(mask):
                        np.add.at(
                            xy_counts[:, class_index],
                            xy_states[mask].ravel(),
                            1,
                        )
                if pair_i:
                    pair_states = (
                        pair_offsets_array[None, :]
                        + X_batch[:, pair_i_array]
                        * self.cardinalities_[pair_j_array][None, :]
                        + X_batch[:, pair_j_array]
                    )
                    for class_index in range(n_classes):
                        mask = y_batch == class_index
                        if np.any(mask):
                            np.add.at(
                                pair_counts[:, class_index],
                                pair_states[mask].ravel(),
                                1,
                            )

        xy_by_feature = [
            xy_counts[
                int(xy_offsets[f]) : int(xy_offsets[f] + self.cardinalities_[f])
            ]
            for f in range(n_features)
        ]
        total = float(len(X))
        mi = np.zeros(n_features, dtype=np.float64)
        for feature, counts in enumerate(xy_by_feature):
            value_counts = counts.sum(axis=1)
            denominator = value_counts[:, None] * class_counts[None, :]
            mask = counts > 0
            mi[feature] = np.sum(
                (counts[mask] / total)
                * np.log2((counts[mask] * total) / denominator[mask])
            )

        cmi = np.zeros((n_features, n_features), dtype=np.float64)
        pair_lookup: dict[tuple[int, int], int] = {}
        for pair_index, (i, j, offset) in enumerate(
            zip(pair_i, pair_j, pair_offsets)
        ):
            pair_lookup[(i, j)] = pair_index
            size = int(self.cardinalities_[i] * self.cardinalities_[j])
            counts = pair_counts[offset : offset + size].reshape(
                int(self.cardinalities_[i]),
                int(self.cardinalities_[j]),
                n_classes,
            )
            denominator = (
                xy_by_feature[i][:, None, :] * xy_by_feature[j][None, :, :]
            )
            class_numerator = counts * class_counts[None, None, :]
            mask = counts > 0
            score = np.sum(
                (counts[mask] / total)
                * np.log2(class_numerator[mask] / denominator[mask])
            )
            cmi[i, j] = cmi[j, i] = score

        self.feature_mi_ = mi
        self.feature_order_ = np.argsort(-mi, kind="mergesort").astype(np.int64)
        self.parents_ = []
        for position, feature in enumerate(self.feature_order_):
            parent_count = min(position, self.k)
            if parent_count == 0:
                self.parents_.append(np.empty(0, dtype=np.int64))
                continue
            candidates = self.feature_order_[:position]
            candidate_scores = cmi[int(feature), candidates]
            ranking = np.argsort(-candidate_scores, kind="mergesort")
            self.parents_.append(candidates[ranking[:parent_count]].copy())

        # Structure statistics are intentionally local: they are not model state.

    def _fit_parameters(self, X: np.ndarray, y: np.ndarray) -> None:
        self.class_counts_ = np.zeros(self.n_classes_, dtype=np.int64)
        self.tables_: list[list[dict[tuple[int, ...], np.ndarray]]] = []
        for position, feature in enumerate(self.feature_order_):
            self.tables_.append([{} for _ in range(len(self.parents_[position]) + 1)])

        with self.pass_tracker_.iter_rows("kdb_parameters", X, y) as rows:
            for row, target in rows:
                self.class_counts_[target] += 1
                for position, feature in enumerate(self.feature_order_):
                    x_value = int(row[feature])
                    parents = self.parents_[position]
                    for depth in range(len(parents) + 1):
                        context = tuple(int(row[p]) for p in parents[:depth])
                        table = self.tables_[position][depth].get(context)
                        if table is None:
                            table = np.zeros(
                                (int(self.cardinalities_[feature]), self.n_classes_),
                                dtype=np.int64,
                            )
                            self.tables_[position][depth][context] = table
                        table[x_value, target] += 1

    @property
    def learner_passes_(self) -> int:
        return self.pass_tracker_.pass_count

    @property
    def learner_rows_processed_(self) -> int:
        return self.pass_tracker_.rows_processed

    @property
    def selected_feature_indices_(self) -> np.ndarray:
        return self.feature_order_[: self.selected_feature_count_].copy()

    def pass_records(self) -> list[dict[str, object]]:
        return self.pass_tracker_.as_records()

    def model_state_bytes(self) -> int:
        return _deep_sizeof(self)

    def predict_proba(self, X: Sequence) -> np.ndarray:
        self._require_fitted()
        X_array = _categorical_matrix(X)
        if X_array.shape[1] != self.n_features_in_:
            raise ValueError("Prediction data has a different feature count")
        probabilities = np.empty((len(X_array), self.n_classes_), dtype=np.float64)
        for row_index, row in enumerate(X_array):
            probabilities[row_index] = _softmax(self._generative_log_scores(row))
        return probabilities

    def predict(self, X: Sequence) -> np.ndarray:
        probabilities = self.predict_proba(X)
        return self.classes_[np.argmax(probabilities, axis=1)]

    def _generative_log_scores(self, row: np.ndarray) -> np.ndarray:
        scores = self._class_log_theta()
        for position in range(self.selected_feature_count_):
            feature = int(self.feature_order_[position])
            table, _, _ = self._lookup_table(row, position, self.selected_k_)
            scores += self._conditional_log_theta(table, int(row[feature]))
        return scores

    def _class_log_theta(self) -> np.ndarray:
        m = self.m_estimate
        return np.log(
            (self.class_counts_ + m / self.n_classes_)
            / (self.n_train_rows_ + m)
        )

    def _conditional_log_theta(
        self, table: np.ndarray, value: int
    ) -> np.ndarray:
        m = self.m_estimate
        cardinality = table.shape[0]
        denominator = table.sum(axis=0)
        return np.log(
            (table[value] + m / cardinality) / (denominator + m)
        )

    def _lookup_table(
        self, row: np.ndarray, position: int, requested_k: int
    ) -> tuple[np.ndarray, int, tuple[int, ...]]:
        parents = self.parents_[position]
        depth = min(int(requested_k), len(parents))
        while depth >= 0:
            context = tuple(int(row[p]) for p in parents[:depth])
            table = self.tables_[position][depth].get(context)
            if table is not None:
                return table, depth, context
            depth -= 1
        raise RuntimeError("Root KDB table is missing")

    def _require_fitted(self) -> None:
        if not hasattr(self, "tables_"):
            raise RuntimeError("Classifier must be fitted before prediction")


### Published selective KDB (sKDB)


In [5]:
class NaiveBayesClassifier(KDBClassifier):
    """Named convenience wrapper for the KDB(k=0) implementation."""

    model_name = "Naive Bayes (KDB k=0)"

    def __init__(self, *, m_estimate: float = 1.0) -> None:
        super().__init__(k=0, m_estimate=m_estimate)


class SelectiveKDBClassifier(KDBClassifier):
    """Published sKDB third-pass incremental LOOCV-RMSE selection."""

    model_name = "sKDB"

    def __init__(self, kmax: int = 5, *, m_estimate: float = 1.0) -> None:
        if int(kmax) != kmax or kmax < 1:
            raise ValueError("kmax must be a positive integer for three-pass sKDB")
        self.kmax = int(kmax)
        super().__init__(k=self.kmax, m_estimate=m_estimate)

    def fit(self, X: Sequence, y: Sequence) -> "SelectiveKDBClassifier":
        X_array = _categorical_matrix(X)
        super().fit(X_array, y)
        y_encoded = np.asarray(
            [self._label_to_index_[label] for label in np.asarray(y, dtype=object)],
            dtype=np.int64,
        )
        self._select_with_loocv(X_array, y_encoded)
        self.training_seconds_ = float(
            sum(record.seconds for record in self.pass_tracker_.records)
        )
        return self

    def _select_with_loocv(self, X: np.ndarray, y: np.ndarray) -> None:
        n_features = self.n_features_in_
        loss = np.zeros((self.kmax + 1, n_features + 1), dtype=np.float64)

        with self.pass_tracker_.iter_rows("skdb_incremental_loocv", X, y) as rows:
            for row, target in rows:
                prior = self._loo_class_log_theta(target)
                prior_probability = _softmax(prior)[target]
                prior_error = (1.0 - prior_probability) ** 2
                loss[:, n_features] += prior_error
                candidate_scores = np.repeat(
                    prior[None, :], self.kmax + 1, axis=0
                )
                for position in range(n_features):
                    for candidate_k in range(self.kmax + 1):
                        candidate_scores[candidate_k] += self._loo_conditional_log_theta(
                            row, target, position, candidate_k
                        )
                        probability = _softmax(candidate_scores[candidate_k])[target]
                        loss[candidate_k, position] += (1.0 - probability) ** 2

        rmse = np.sqrt(loss / self.n_train_rows_)
        # Fupla starts at prior-only, retains kmax if no prefix improves it, and
        # uses strict '<' while scanning feature prefix first, then k.
        best_rmse = float(rmse[0, n_features])
        best_k = self.kmax
        best_feature_count = 0
        for position in range(n_features):
            for candidate_k in range(self.kmax + 1):
                candidate = float(rmse[candidate_k, position])
                if candidate < best_rmse:
                    best_rmse = candidate
                    best_k = candidate_k
                    best_feature_count = position + 1

        self.selection_rmse_ = rmse
        self.selection_best_rmse_ = best_rmse
        self.selected_k_ = int(best_k)
        self.selected_feature_count_ = int(best_feature_count)
        self.parents_ = [
            parents[: self.selected_k_].copy()
            for parents in self.parents_[: self.selected_feature_count_]
        ]
        self.tables_ = [
            depths[: self.selected_k_ + 1]
            for depths in self.tables_[: self.selected_feature_count_]
        ]

    def _loo_class_log_theta(self, target: int) -> np.ndarray:
        counts = self.class_counts_.astype(np.float64).copy()
        counts[target] -= 1.0
        m = self.m_estimate
        return np.log(
            (counts + m / self.n_classes_) / (self.n_train_rows_ - 1 + m)
        )

    def _loo_conditional_log_theta(
        self,
        row: np.ndarray,
        target: int,
        position: int,
        candidate_k: int,
    ) -> np.ndarray:
        parents = self.parents_[position]
        depth = min(candidate_k, len(parents))
        feature = int(self.feature_order_[position])
        value = int(row[feature])
        while depth > 0:
            context = tuple(int(row[p]) for p in parents[:depth])
            table = self.tables_[position][depth].get(context)
            # Fupla's LOOCV minCount=2 test is on this target feature value
            # across classes before the current row is discounted.
            if table is not None and int(table[value].sum()) >= 2:
                break
            depth -= 1
        context = tuple(int(row[p]) for p in parents[:depth])
        table = self.tables_[position][depth][context]
        numerator = table[value].astype(np.float64).copy()
        denominator = table.sum(axis=0).astype(np.float64)
        numerator[target] -= 1.0
        denominator[target] -= 1.0
        m = self.m_estimate
        return np.log(
            (numerator + m / table.shape[0]) / (denominator + m)
        )


### dKDB and experimental/unpublished sdKDB


In [6]:
class _WeightedBNMixin:
    """Streaming discriminative weights over fixed generative log-theta."""

    optimizer: str
    learning_rate: float
    discriminative_passes: int
    adagrad_epsilon: float

    def _validate_discriminative_options(self) -> None:
        if self.optimizer not in {"sgd", "adagrad"}:
            raise ValueError("optimizer must be 'sgd' or 'adagrad'")
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be positive")
        if int(self.discriminative_passes) != self.discriminative_passes:
            raise ValueError("discriminative_passes must be an integer")
        if self.discriminative_passes < 0:
            raise ValueError("discriminative_passes must be non-negative")
        if self.adagrad_epsilon <= 0:
            raise ValueError("adagrad_epsilon must be positive")

    def _fit_weighted_bn(self, X: np.ndarray, y: np.ndarray) -> None:
        self.discriminative_iterations_ = int(self.discriminative_passes)
        self.discriminative_updates_ = 0
        self.generative_theta_checksum_before_ = self._generative_checksum()
        if self.discriminative_passes == 0:
            self.class_weights_ = None
            self.factor_weights_ = None
            self.generative_theta_checksum_after_ = self._generative_checksum()
            return

        self.class_weights_ = np.zeros(self.n_classes_, dtype=np.float64)
        self.factor_weights_: dict[
            tuple[int, int, tuple[int, ...]], np.ndarray
        ] = {}
        class_accumulator = np.zeros_like(self.class_weights_)
        factor_accumulators: dict[
            tuple[int, int, tuple[int, ...]], np.ndarray
        ] = {}

        for iteration in range(self.discriminative_passes):
            stage = f"weighted_bn_{self.optimizer}_{iteration + 1}"
            with self.pass_tracker_.iter_rows(stage, X, y) as rows:
                for row, target in rows:
                    class_log_theta = self._class_log_theta()
                    active_factors = []
                    scores = self.class_weights_ * class_log_theta
                    for position in range(self.selected_feature_count_):
                        feature = int(self.feature_order_[position])
                        value = int(row[feature])
                        table, depth, context = self._lookup_table(
                            row, position, self.selected_k_
                        )
                        key = (position, depth, context)
                        weights = self.factor_weights_.get(key)
                        if weights is None:
                            weights = np.zeros(table.shape, dtype=np.float64)
                            self.factor_weights_[key] = weights
                            factor_accumulators[key] = np.zeros(
                                table.shape, dtype=np.float64
                            )
                        log_theta = self._conditional_log_theta(table, value)
                        scores += weights[value] * log_theta
                        active_factors.append((key, value, log_theta))

                    probabilities = _softmax(scores)
                    residual = probabilities.copy()
                    residual[target] -= 1.0

                    class_gradient = residual * class_log_theta
                    self._update_vector(
                        self.class_weights_,
                        class_accumulator,
                        class_gradient,
                    )
                    for key, value, log_theta in active_factors:
                        gradient = residual * log_theta
                        self._update_vector(
                            self.factor_weights_[key][value],
                            factor_accumulators[key][value],
                            gradient,
                        )
                    self.discriminative_updates_ += 1

        self.generative_theta_checksum_after_ = self._generative_checksum()
        if (
            self.generative_theta_checksum_before_
            != self.generative_theta_checksum_after_
        ):
            raise AssertionError("Discriminative learning modified generative theta")

    def _update_vector(
        self,
        parameters: np.ndarray,
        accumulator: np.ndarray,
        gradient: np.ndarray,
    ) -> None:
        if self.optimizer == "sgd":
            parameters -= self.learning_rate * gradient
            return
        accumulator += gradient * gradient
        parameters -= (
            self.learning_rate
            / (self.adagrad_epsilon + np.sqrt(accumulator))
        ) * gradient

    def _generative_checksum(self) -> str:
        state = (self.class_counts_, self.tables_, self.parents_)
        return sha256(pickle.dumps(state, protocol=pickle.HIGHEST_PROTOCOL)).hexdigest()

    def predict_proba(self, X: Sequence) -> np.ndarray:
        if self.discriminative_passes == 0:
            return super().predict_proba(X)
        self._require_fitted()
        X_array = _categorical_matrix(X)
        probabilities = np.empty((len(X_array), self.n_classes_), dtype=np.float64)
        for row_index, row in enumerate(X_array):
            class_log_theta = self._class_log_theta()
            scores = self.class_weights_ * class_log_theta
            for position in range(self.selected_feature_count_):
                feature = int(self.feature_order_[position])
                value = int(row[feature])
                table, depth, context = self._lookup_table(
                    row, position, self.selected_k_
                )
                weights = self.factor_weights_.get((position, depth, context))
                if weights is not None:
                    scores += weights[value] * self._conditional_log_theta(
                        table, value
                    )
            probabilities[row_index] = _softmax(scores)
        return probabilities


class DiscriminativeKDBClassifier(_WeightedBNMixin, KDBClassifier):
    """dKDB: KDB theta followed by streaming weighted-BN optimization."""

    model_name = "dKDB"

    def __init__(
        self,
        k: int = 1,
        *,
        discriminative_passes: int = 1,
        optimizer: str = "adagrad",
        learning_rate: float = 0.01,
        adagrad_epsilon: float = 1e-9,
        m_estimate: float = 1.0,
    ) -> None:
        super().__init__(k=k, m_estimate=m_estimate)
        self.discriminative_passes = int(discriminative_passes)
        self.optimizer = optimizer.lower()
        self.learning_rate = float(learning_rate)
        self.adagrad_epsilon = float(adagrad_epsilon)
        self._validate_discriminative_options()

    def fit(self, X: Sequence, y: Sequence) -> "DiscriminativeKDBClassifier":
        X_array = _categorical_matrix(X)
        super().fit(X_array, y)
        y_encoded = np.asarray(
            [self._label_to_index_[label] for label in np.asarray(y, dtype=object)],
            dtype=np.int64,
        )
        self._fit_weighted_bn(X_array, y_encoded)
        self.training_seconds_ = float(
            sum(record.seconds for record in self.pass_tracker_.records)
        )
        return self


class SelectiveDiscriminativeKDBClassifier(
    _WeightedBNMixin, SelectiveKDBClassifier
):
    """Experimental/unpublished sdKDB: Fupla ``-S`` selection then ``-D``.

    No extra selection or optimization stage is added beyond published sKDB
    followed by the same weighted-BN stage used by dKDB.
    """

    model_name = "sdKDB (experimental/unpublished)"

    def __init__(
        self,
        kmax: int = 5,
        *,
        discriminative_passes: int = 1,
        optimizer: str = "adagrad",
        learning_rate: float = 0.01,
        adagrad_epsilon: float = 1e-9,
        m_estimate: float = 1.0,
    ) -> None:
        super().__init__(kmax=kmax, m_estimate=m_estimate)
        self.discriminative_passes = int(discriminative_passes)
        self.optimizer = optimizer.lower()
        self.learning_rate = float(learning_rate)
        self.adagrad_epsilon = float(adagrad_epsilon)
        self._validate_discriminative_options()

    def fit(
        self, X: Sequence, y: Sequence
    ) -> "SelectiveDiscriminativeKDBClassifier":
        X_array = _categorical_matrix(X)
        super().fit(X_array, y)
        y_encoded = np.asarray(
            [self._label_to_index_[label] for label in np.asarray(y, dtype=object)],
            dtype=np.int64,
        )
        self._fit_weighted_bn(X_array, y_encoded)
        self.training_seconds_ = float(
            sum(record.seconds for record in self.pass_tracker_.records)
        )
        return self


### Shared frozen model helpers


In [7]:
def _categorical_matrix(X: Sequence) -> np.ndarray:
    if hasattr(X, "to_numpy"):
        X = X.to_numpy(copy=False)
    array = np.asarray(X)
    if array.ndim != 2:
        raise ValueError("X must be a two-dimensional matrix")
    if not np.issubdtype(array.dtype, np.integer):
        if not np.all(np.isfinite(array)) or not np.all(array == np.floor(array)):
            raise TypeError("KDB learners require discrete integer features")
    array = array.astype(np.int64, copy=False)
    if np.any(array < 0):
        raise ValueError("Discrete feature states must be non-negative")
    return array


def _encode_labels(
    y: Sequence,
) -> tuple[np.ndarray, np.ndarray, dict[object, int]]:
    labels = np.asarray(y, dtype=object).reshape(-1)
    mapping: dict[object, int] = {}
    encoded = np.empty(len(labels), dtype=np.int64)
    classes: list[object] = []
    for index, label in enumerate(labels.tolist()):
        try:
            encoded[index] = mapping[label]
        except KeyError:
            mapping[label] = len(classes)
            encoded[index] = len(classes)
            classes.append(label)
    return encoded, np.asarray(classes, dtype=object), mapping


def _softmax(log_scores: np.ndarray) -> np.ndarray:
    shifted = log_scores - np.max(log_scores)
    probabilities = np.exp(shifted)
    total = probabilities.sum()
    if not np.isfinite(total) or total <= 0:
        return np.full(len(log_scores), 1.0 / len(log_scores))
    return probabilities / total


def _deep_sizeof(value: object, seen: set[int] | None = None) -> int:
    """Approximate live Python model-state memory without double counting."""

    if seen is None:
        seen = set()
    identity = id(value)
    if identity in seen:
        return 0
    seen.add(identity)
    size = sys.getsizeof(value)
    if isinstance(value, dict):
        size += sum(
            _deep_sizeof(key, seen) + _deep_sizeof(item, seen)
            for key, item in value.items()
        )
    elif isinstance(value, (list, tuple, set, frozenset)):
        size += sum(_deep_sizeof(item, seen) for item in value)
    elif hasattr(value, "__dict__"):
        size += _deep_sizeof(vars(value), seen)
    return int(size)


## 4. Correctness tests (defined before any real-data benchmark)


In [8]:
def headline_model_factories(kmax=KMAX) -> Mapping[str, Callable[[], object]]:
    """The four-model, structurally matched research comparison."""
    return {
        "KDB": lambda: KDBClassifier(k=kmax),
        "sKDB": lambda: SelectiveKDBClassifier(kmax=kmax),
        "dKDB": lambda: DiscriminativeKDBClassifier(
            k=kmax, discriminative_passes=D, optimizer=OPTIMIZER,
            learning_rate=LEARNING_RATE, adagrad_epsilon=ADAGRAD_EPSILON,
        ),
        "sdKDB (experimental/unpublished)": lambda: SelectiveDiscriminativeKDBClassifier(
            kmax=kmax, discriminative_passes=D, optimizer=OPTIMIZER,
            learning_rate=LEARNING_RATE, adagrad_epsilon=ADAGRAD_EPSILON,
        ),
    }


def sensitivity_model_factories() -> Mapping[str, Callable[[], object]]:
    """Separate k=0...5 KDB sensitivity experiment; not headline results."""
    return {
        f"KDB(k={candidate_k})": (
            lambda candidate_k=candidate_k: KDBClassifier(k=candidate_k)
        )
        for candidate_k in SENSITIVITY_K_VALUES
    }


def multiclass_data():
    rng = np.random.default_rng(17)
    y = np.repeat(np.arange(3), 18)
    X = np.column_stack([
        y,
        (y + rng.integers(0, 2, len(y))) % 3,
        rng.integers(0, 4, len(y)),
        (2 * y + rng.integers(0, 3, len(y))) % 5,
    ]).astype(np.int64)
    order = rng.permutation(len(y))
    return X[order], y[order]


def assert_normalized(model, X):
    probabilities = model.predict_proba(X)
    assert np.all(np.isfinite(probabilities))
    assert np.all((probabilities >= 0.0) & (probabilities <= 1.0))
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-12)


def test_headline_capacity_is_matched_at_five():
    factories = headline_model_factories()
    assert factories["KDB"]().k == 5
    assert factories["sKDB"]().kmax == 5
    assert factories["dKDB"]().k == 5
    assert factories["sdKDB (experimental/unpublished)"]().kmax == 5


def test_kdb_k_zero_is_true_one_pass_naive_bayes():
    X, y = multiclass_data()
    nb = NaiveBayesClassifier().fit(X, y)
    kdb_zero = KDBClassifier(k=0).fit(X, y)
    np.testing.assert_allclose(nb.predict_proba(X), kdb_zero.predict_proba(X), atol=1e-12)
    assert kdb_zero.learner_passes_ == 1
    assert kdb_zero.learner_rows_processed_ == len(X)
    assert not kdb_zero.structure_learned_
    assert not hasattr(kdb_zero, "feature_mi_")


def test_all_model_probabilities_are_normalized_and_multiclass_safe():
    X, y = multiclass_data()
    for model in headline_model_factories().values():
        fitted = model().fit(X, y)
        assert fitted.n_classes_ == 3
        assert_normalized(fitted, X)


def test_pass_counts_and_rows_are_exact():
    X, y = multiclass_data()
    cases = [
        (KDBClassifier(k=0), 1),
        (KDBClassifier(k=5), 2),
        (SelectiveKDBClassifier(kmax=5), 3),
        (DiscriminativeKDBClassifier(k=5, discriminative_passes=2), 4),
        (SelectiveDiscriminativeKDBClassifier(kmax=5, discriminative_passes=2), 5),
    ]
    for model, expected_passes in cases:
        model.fit(X, y)
        assert model.learner_passes_ == expected_passes
        assert model.learner_rows_processed_ == expected_passes * len(X)
        assert len(model.pass_records()) == expected_passes


def test_skdb_selected_k_is_bounded_by_kmax():
    X, y = multiclass_data()
    model = SelectiveKDBClassifier(kmax=5).fit(X, y)
    assert 0 <= model.selected_k_ <= model.kmax
    assert 0 <= model.selected_feature_count_ <= X.shape[1]
    assert model.selection_rmse_.shape == (model.kmax + 1, X.shape[1] + 1)


def test_dkdb_zero_discriminative_passes_equals_kdb():
    X, y = multiclass_data()
    kdb = KDBClassifier(k=5).fit(X, y)
    dkdb = DiscriminativeKDBClassifier(k=5, discriminative_passes=0).fit(X, y)
    np.testing.assert_allclose(kdb.predict_proba(X), dkdb.predict_proba(X), atol=1e-12)
    assert dkdb.learner_passes_ == kdb.learner_passes_


def test_sdkdb_zero_discriminative_passes_equals_skdb():
    X, y = multiclass_data()
    skdb = SelectiveKDBClassifier(kmax=5).fit(X, y)
    sdkdb = SelectiveDiscriminativeKDBClassifier(kmax=5, discriminative_passes=0).fit(X, y)
    assert skdb.selected_k_ == sdkdb.selected_k_
    assert skdb.selected_feature_count_ == sdkdb.selected_feature_count_
    np.testing.assert_allclose(skdb.predict_proba(X), sdkdb.predict_proba(X), atol=1e-12)
    assert sdkdb.learner_passes_ == skdb.learner_passes_


def test_discriminative_stage_keeps_generative_theta_fixed():
    X, y = multiclass_data()
    model = DiscriminativeKDBClassifier(k=5, discriminative_passes=2, optimizer="adagrad").fit(X, y)
    assert model.generative_theta_checksum_before_ == model.generative_theta_checksum_after_
    assert model.discriminative_updates_ == 2 * len(X)


## 5. Dataset registry, loaders, and validation


All loaders return `(X, y, metadata)`. The registry uses pinned OpenML
IDs or named primary archives. YearPredictionMSD is intentionally treated
as multiclass using distinct year values solely to reproduce the KDB-paper
treatment; UCI defines the original task as regression. SUSY and HIGGS
preserve their conventional final-500k held-out test blocks in STRESS mode.


In [9]:
DATASET_REGISTRY = {
    "Skin Segmentation": {
        "source": "UCI Skin Segmentation via OpenML data_id=1502",
        "url": "https://www.openml.org/d/1502", "group": "core",
    },
    "MiniBooNE": {
        "source": "UCI MiniBooNE via OpenML data_id=41150",
        "url": "https://www.openml.org/d/41150", "group": "core",
    },
    "cod-rna": {
        "source": "LIBSVM cod-rna train/validation/test files",
        "url": "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary.html#cod-rna", "group": "core",
    },
    "Covertype": {
        "source": "UCI Covertype via sklearn fetch_covtype",
        "url": "https://archive.ics.uci.edu/dataset/31/covertype", "group": "core",
    },
    "YearPredictionMSD": {
        "source": "UCI YearPredictionMSD archive; years reinterpreted as class labels for paper alignment",
        "url": "https://archive.ics.uci.edu/dataset/203/yearpredictionmsd", "group": "paper_multiclass",
        "original_task": "regression", "benchmark_task": "multiclass classification (distinct years)",
        "predefined_test_start": 463_715,
    },
    "SUSY": {
        "source": "UCI SUSY archive",
        "url": "https://archive.ics.uci.edu/dataset/279/susy", "group": "stress",
        "predefined_test_start": 4_500_000,
    },
    "HIGGS": {
        "source": "UCI HIGGS archive",
        "url": "https://archive.ics.uci.edu/dataset/280/higgs", "group": "stress",
        "predefined_test_start": 10_500_000,
    },
}

UCI_FILES = {
    "YearPredictionMSD": "https://archive.ics.uci.edu/static/public/203/yearpredictionmsd.zip",
    "SUSY": "https://archive.ics.uci.edu/static/public/279/susy.zip",
    "HIGGS": "https://archive.ics.uci.edu/static/public/280/higgs.zip",
}
COD_RNA_FILES = {
    "train": "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary/cod-rna",
    "validation": "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary/cod-rna.t",
    "remaining": "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary/cod-rna.r",
}


def _download(url, filename):
    destination = DATA_CACHE_DIR / filename
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        return destination

    temporary = destination.with_suffix(destination.suffix + ".part")

    print(f"Downloading {url} -> {destination}")

    ssl_context = ssl.create_default_context(
        cafile=certifi.where()
    )

    with urllib.request.urlopen(
        url,
        context=ssl_context
    ) as response:
        with open(temporary, "wb") as f:
            shutil.copyfileobj(response, f)

    temporary.replace(destination)

    return destination

def _numeric_frame(X: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    converted = pd.DataFrame(index=X.index)
    for column in X.columns:
        try:
            converted[str(column)] = pd.to_numeric(X[column], errors="raise")
        except (TypeError, ValueError) as exc:
            raise TypeError(
                f"{dataset_name}: feature {column!r} is not numeric. "
                "Arbitrary categorical variables are not ordinal-encoded."
            ) from exc
    return converted


def _stratified_limit(X, y, max_rows, seed):
    if max_rows is None or max_rows >= len(X):
        return X.reset_index(drop=True), y.reset_index(drop=True), False
    if max_rows < 2 * y.nunique():
        raise ValueError("Row limit is too small for stratified sampling")
    X_sample, _, y_sample, _ = train_test_split(
        X, y, train_size=max_rows, stratify=y, random_state=seed,
    )
    return X_sample.reset_index(drop=True), y_sample.reset_index(drop=True), True


def _load_openml(data_id: int, dataset_name: str):
    bunch = fetch_openml(data_id=data_id, as_frame=True, parser="auto", data_home=DATA_CACHE_DIR)
    return _numeric_frame(bunch.data, dataset_name), pd.Series(bunch.target, name="target")


def _load_cod_rna():
    matrices, labels = [], []
    for part, url in COD_RNA_FILES.items():
        path = _download(url, f"cod-rna.{part}.libsvm")
        X_part, y_part = load_svmlight_file(path, n_features=8)
        matrices.append(X_part.toarray())
        labels.append(y_part)
    X = pd.DataFrame(np.vstack(matrices), columns=[f"feature_{i}" for i in range(8)])
    y = pd.Series(np.concatenate(labels).astype(np.int8), name="target")
    return X, y


def _zip_csv_member(path: Path, member_hint: str, **read_csv_kwargs):
    with zipfile.ZipFile(path) as archive:
        candidates = [name for name in archive.namelist() if member_hint.lower() in name.lower()]
        if not candidates:
            candidates = [name for name in archive.namelist() if not name.endswith("/")]
        if len(candidates) != 1:
            raise RuntimeError(f"Expected one matching data member in {path}; found {candidates}")
        with archive.open(candidates[0]) as stream:
            if candidates[0].lower().endswith(".gz"):
                with gzip.GzipFile(fileobj=stream) as decompressed:
                    return pd.read_csv(decompressed, **read_csv_kwargs)
            return pd.read_csv(stream, **read_csv_kwargs)


def _load_year_prediction():
    path = _download(UCI_FILES["YearPredictionMSD"], "YearPredictionMSD.zip")
    frame = _zip_csv_member(path, "YearPredictionMSD.txt", header=None)
    y = frame.iloc[:, 0].astype(np.int32).rename("year_class")
    X = frame.iloc[:, 1:].copy()
    X.columns = [f"feature_{i}" for i in range(X.shape[1])]
    return X, y


def _load_stress_csv(dataset_name: str):
    path = _download(UCI_FILES[dataset_name], f"{dataset_name}.zip")
    frame = _zip_csv_member(path, f"{dataset_name}.csv", header=None)
    y = frame.iloc[:, 0].astype(np.int8).rename("target")
    X = frame.iloc[:, 1:].copy()
    X.columns = [f"feature_{i}" for i in range(X.shape[1])]
    return X, y


def validate_dataset(X, y, metadata):
    X = _numeric_frame(pd.DataFrame(X), metadata["name"])
    y = pd.Series(y, name="target").reset_index(drop=True)
    X = X.reset_index(drop=True)
    if len(X) != len(y) or len(X) == 0 or X.shape[1] == 0:
        raise ValueError("Dataset must contain aligned nonempty X and y")
    if y.isna().any():
        raise ValueError(f"{metadata['name']}: target contains missing values")
    array = X.to_numpy(dtype=np.float64, copy=False)
    counts = y.value_counts(dropna=False).sort_index()
    missing = int(X.isna().sum().sum())
    nonfinite = int(np.size(array) - np.isfinite(array).sum())
    metadata = dict(metadata)
    metadata.update({
        "n_rows": int(len(X)), "n_features": int(X.shape[1]),
        "n_classes": int(y.nunique()),
        "class_counts": json.dumps({str(k): int(v) for k, v in counts.items()}),
        "class_proportions": json.dumps({str(k): float(v / len(y)) for k, v in counts.items()}),
        "missing_feature_values": missing, "nonfinite_feature_values": nonfinite,
        "numeric_dtype_compatible": True,
        "feature_dtypes": json.dumps({str(k): str(v) for k, v in X.dtypes.items()}),
        "feature_dtype_summary": json.dumps(X.dtypes.astype(str).value_counts().to_dict()),
        "approx_raw_memory_bytes": int(X.memory_usage(index=True, deep=True).sum() + y.memory_usage(index=True, deep=True)),
    })
    return X, y, metadata


def load_dataset(dataset_name: str, *, max_rows: int | None = None, seed: int = RANDOM_SEED):
    """Return numeric X, y and fully audited metadata."""
    if dataset_name not in DATASET_REGISTRY:
        raise KeyError(f"Unknown dataset {dataset_name!r}")
    if dataset_name == "Skin Segmentation":
        X, y = _load_openml(1502, dataset_name)
    elif dataset_name == "MiniBooNE":
        X, y = _load_openml(41150, dataset_name)
    elif dataset_name == "cod-rna":
        X, y = _load_cod_rna()
    elif dataset_name == "Covertype":
        bunch = fetch_covtype(data_home=DATA_CACHE_DIR, as_frame=True)
        X, y = _numeric_frame(bunch.data, dataset_name), pd.Series(bunch.target, name="target")
    elif dataset_name == "YearPredictionMSD":
        X, y = _load_year_prediction()
    elif dataset_name in {"SUSY", "HIGGS"}:
        X, y = _load_stress_csv(dataset_name)
    else:
        raise AssertionError("Registry/loader mismatch")

    full_rows = len(X)
    X, y, sampled = _stratified_limit(X, pd.Series(y), max_rows, seed)
    metadata = {
        "name": dataset_name, **DATASET_REGISTRY[dataset_name],
        "full_rows_before_mode_limit": int(full_rows),
        "sampled": bool(sampled), "requested_row_limit": max_rows,
        "sample_seed": seed if sampled else None,
    }
    if sampled:
        metadata["predefined_test_start"] = None
    return validate_dataset(X, y, metadata)


def metadata_display_row(metadata):
    columns = [
        "name", "source", "url", "n_rows", "n_features", "n_classes",
        "class_counts", "class_proportions", "missing_feature_values",
        "nonfinite_feature_values", "numeric_dtype_compatible", "feature_dtype_summary",
        "approx_raw_memory_bytes", "sampled", "requested_row_limit",
    ]
    return {column: metadata.get(column) for column in columns}


## 6. Evaluation, metrics, pass accounting, and resumability


In [10]:
@dataclass(frozen=True)
class EvaluationConfig:
    rounds: int = 2
    folds: int = 2
    random_state: int = RANDOM_SEED
    include_binary_metrics: bool = True
    positive_class: object | None = None

    def validate(self, n_classes: int, *, custom_splits: bool = False):
        if self.rounds < 1:
            raise ValueError("rounds must be >=1")
        if not custom_splits and self.folds < 2:
            raise ValueError("folds must be >=2 for external CV")
        if n_classes < 2:
            raise ValueError("Classification requires at least two classes")


@dataclass
class EvaluationResult:
    run_results: pd.DataFrame
    pass_audit: pd.DataFrame

    @property
    def fold_results(self):
        return self.run_results

    @property
    def summary(self):
        return per_dataset_model_summary(self.run_results)


def _json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(type(value).__name__)


class ResultCache:
    """Append-only per-configuration run/pass records for exact resume."""

    def __init__(self, root: Path, configuration_id: str, enabled: bool = True):
        self.enabled = enabled
        self.directory = root / configuration_id
        self.run_file = self.directory / "run_records.jsonl"
        self.pass_file = self.directory / "pass_records.jsonl"
        self.runs = self._read(self.run_file)
        self.passes = self._read(self.pass_file)
        self.runs_by_key = {row["cache_key"]: row for row in self.runs}
        self.passes_by_key = {}
        for row in self.passes:
            self.passes_by_key.setdefault(row["cache_key"], []).append(row)

    @staticmethod
    def _read(path):
        if not path.exists():
            return []
        records = []
        with path.open("r", encoding="utf-8") as stream:
            for line in stream:
                if line.strip():
                    records.append(json.loads(line))
        return records

    def get(self, key):
        if not (self.enabled and RESUME_COMPLETED_RUNS):
            return None
        row = self.runs_by_key.get(key)
        if row is None:
            return None
        return row, self.passes_by_key.get(key, [])

    def append(self, run_row, pass_rows):
        if not self.enabled:
            return
        self.directory.mkdir(parents=True, exist_ok=True)
        with self.run_file.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(run_row, default=_json_default, sort_keys=True) + "\n")
        with self.pass_file.open("a", encoding="utf-8") as stream:
            for row in pass_rows:
                stream.write(json.dumps(row, default=_json_default, sort_keys=True) + "\n")
        self.runs_by_key[run_row["cache_key"]] = run_row
        self.passes_by_key[run_row["cache_key"]] = list(pass_rows)


def _matrix(X):
    if hasattr(X, "to_numpy"):
        X = X.to_numpy(copy=False)
    array = np.asarray(X)
    if array.ndim != 2:
        raise ValueError("X must be a two-dimensional matrix")
    return array


def _encode_global_labels(labels):
    mapping, classes = {}, []
    encoded = np.empty(len(labels), dtype=np.int64)
    for index, label in enumerate(np.asarray(labels, dtype=object).tolist()):
        if label not in mapping:
            mapping[label] = len(classes)
            classes.append(label)
        encoded[index] = mapping[label]
    return encoded, np.asarray(classes, dtype=object)


def _align_probabilities(probabilities, model_classes, n_classes):
    aligned = np.zeros((len(probabilities), n_classes), dtype=np.float64)
    for local_index, class_label in enumerate(model_classes):
        aligned[:, int(class_label)] = probabilities[:, local_index]
    return aligned


def _positive_class_index(classes, positive_class):
    if positive_class is None:
        return 1
    matches = np.flatnonzero(classes == positive_class)
    if len(matches) != 1:
        raise ValueError(f"positive_class {positive_class!r} is not in y")
    return int(matches[0])


def _model_parameter_context_counts(model):
    selected = int(getattr(model, "selected_feature_count_", model.n_features_in_))
    selected_k = int(getattr(model, "selected_k_", getattr(model, "k", 0)))
    contexts = 0
    generative_parameters = int(model.class_counts_.size)
    for position in range(selected):
        max_depth = min(selected_k, len(model.tables_[position]) - 1)
        for depth in range(max_depth + 1):
            contexts += len(model.tables_[position][depth])
            generative_parameters += sum(table.size for table in model.tables_[position][depth].values())
    discriminative_parameters = 0
    if getattr(model, "class_weights_", None) is not None:
        discriminative_parameters += int(model.class_weights_.size)
    if getattr(model, "factor_weights_", None) is not None:
        discriminative_parameters += int(sum(value.size for value in model.factor_weights_.values()))
    return int(generative_parameters + discriminative_parameters), int(contexts)


def _peak_process_rss_bytes():
    """OS high-water RSS where resource.getrusage is available; else NaN."""
    try:
        import resource
        value = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return value if platform.system() == "Darwin" else value * 1024.0
    except (ImportError, AttributeError, OSError):
        return np.nan


def _dataset_fingerprint(X, y, metadata):
    X_array = np.ascontiguousarray(_matrix(X), dtype=np.float64)
    y_array = np.asarray(y, dtype=object)
    probe = np.concatenate([X_array[:3].ravel(), X_array[-3:].ravel()]) if len(X_array) else np.array([])
    payload = {
        "name": metadata.get("name"), "shape": X_array.shape,
        "class_counts": pd.Series(y_array).value_counts().sort_index().to_dict(),
        "probe": probe.tolist(),
    }
    return sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]


def _standard_split_plan(X, y_encoded, config):
    splitter = RepeatedStratifiedKFold(
        n_splits=config.folds, n_repeats=config.rounds,
        random_state=config.random_state,
    )
    plan = []
    for split_index, (train_index, test_index) in enumerate(splitter.split(X, y_encoded)):
        plan.append({
            "split_id": f"repeat={split_index // config.folds + 1},fold={split_index % config.folds + 1}",
            "repeat": split_index // config.folds + 1,
            "fold": split_index % config.folds + 1,
            "train_index": train_index, "test_index": test_index,
            "scaling_train_rows": None,
        })
    return plan


def evaluate_models(
    X, y, model_factories, *, dataset_name, config=None,
    preprocessor_factory=MDLDiscretizer, feature_names=None,
    split_plan=None, metadata=None, cache=None, experiment="headline",
):
    """One row per dataset x split x model; one shared MDL fit per split."""
    if not model_factories:
        raise ValueError("model_factories cannot be empty")
    X_array = _matrix(X)
    labels = np.asarray(y, dtype=object).reshape(-1)
    if len(X_array) != len(labels):
        raise ValueError("X and y must contain the same number of rows")
    y_encoded, classes = _encode_global_labels(labels)
    config = config or EvaluationConfig()
    config.validate(len(classes), custom_splits=split_plan is not None)
    if feature_names is None:
        feature_names = list(map(str, getattr(X, "columns", [f"feature_{i}" for i in range(X_array.shape[1])])) )
    feature_names = list(feature_names)
    if len(feature_names) != X_array.shape[1]:
        raise ValueError("feature_names length does not match X")
    if split_plan is None:
        counts = np.bincount(y_encoded, minlength=len(classes))
        if np.min(counts) < config.folds:
            raise ValueError("Every class needs at least one row in each fold")
        split_plan = _standard_split_plan(X_array, y_encoded, config)
    metadata = metadata or {"name": dataset_name}
    fingerprint = _dataset_fingerprint(X_array, labels, metadata)
    cache = cache or ResultCache(RESULT_CACHE_ROOT, CONFIGURATION_ID, enabled=False)
    run_rows, pass_rows = [], []

    for split in split_plan:
        train_index = np.asarray(split["train_index"], dtype=np.int64)
        test_index = np.asarray(split["test_index"], dtype=np.int64)
        base_key = {
            "configuration_id": CONFIGURATION_ID, "experiment": experiment,
            "dataset": dataset_name, "dataset_fingerprint": fingerprint,
            "split_id": split["split_id"],
        }
        keys = {
            model_name: sha256(json.dumps({**base_key, "model": model_name}, sort_keys=True).encode()).hexdigest()
            for model_name in model_factories
        }
        cached = {name: cache.get(key) for name, key in keys.items()}
        if all(item is not None for item in cached.values()):
            for item in cached.values():
                run_rows.append(item[0]); pass_rows.extend(item[1])
            print(f"RESUME {dataset_name} {split['split_id']}: all models loaded")
            continue

        X_train, X_test = X_array[train_index], X_array[test_index]
        y_train, y_test = y_encoded[train_index], y_encoded[test_index]
        preprocessor = preprocessor_factory()
        preprocessing_started = perf_counter()
        preprocessor.fit(X_train, y_train)
        X_train_discrete = preprocessor.transform(X_train)
        X_test_discrete = preprocessor.transform(X_test)
        preprocessing_seconds = float(perf_counter() - preprocessing_started)
        fit_rows = int(getattr(preprocessor, "fit_rows_", len(train_index)))
        if fit_rows != len(train_index):
            raise AssertionError("Preprocessor fit-row audit does not match fold")

        for model_name, factory in model_factories.items():
            if cached[model_name] is not None:
                run_rows.append(cached[model_name][0]); pass_rows.extend(cached[model_name][1])
                continue
            model = factory()
            rss_before = _peak_process_rss_bytes()
            learner_started = perf_counter()
            model.fit(X_train_discrete, y_train)
            learner_seconds = float(perf_counter() - learner_started)
            prediction_started = perf_counter()
            local_probabilities = model.predict_proba(X_test_discrete)
            prediction_seconds = float(perf_counter() - prediction_started)
            rss_after = _peak_process_rss_bytes()
            probabilities = _align_probabilities(local_probabilities, model.classes_, len(classes))
            if not np.all(np.isfinite(probabilities)):
                raise AssertionError("Model produced non-finite probabilities")
            np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-10)

            predictions = np.argmax(probabilities, axis=1)
            true_probabilities = probabilities[np.arange(len(y_test)), y_test]
            one_hot = np.eye(len(classes), dtype=np.float64)[y_test]
            zero_one = float(np.mean(predictions != y_test))
            rmse = float(np.sqrt(np.mean((1.0 - true_probabilities) ** 2)))
            nll = float(log_loss(y_test, probabilities, labels=np.arange(len(classes))))
            brier = float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))
            roc_auc = average_precision = np.nan
            if config.include_binary_metrics and len(classes) == 2:
                positive = _positive_class_index(classes, config.positive_class)
                target_binary = (y_test == positive).astype(np.int8)
                roc_auc = float(roc_auc_score(target_binary, probabilities[:, positive]))
                average_precision = float(average_precision_score(target_binary, probabilities[:, positive]))

            selected_indices = np.asarray(getattr(model, "selected_feature_indices_", np.arange(X_array.shape[1])), dtype=np.int64)
            selected_names = [feature_names[index] for index in selected_indices]
            configured_k = None if hasattr(model, "kmax") else getattr(model, "k", None)
            kmax = getattr(model, "kmax", None)
            selected_k = int(getattr(model, "selected_k_", configured_k or 0))
            selected_count = int(getattr(model, "selected_feature_count_", X_array.shape[1]))
            passes = int(model.learner_passes_)
            learner_rows = int(model.learner_rows_processed_)
            if learner_rows != passes * len(train_index):
                raise AssertionError("Learner rows do not equal passes x fold rows")
            parameter_count, context_count = _model_parameter_context_counts(model)
            model_pass_rows = []
            for record in model.pass_records():
                model_pass_rows.append({
                    "cache_key": keys[model_name], "configuration_id": CONFIGURATION_ID,
                    "experiment": experiment, "dataset": dataset_name, "model": model_name,
                    "split_id": split["split_id"], "repeat": split.get("repeat"),
                    "fold": split.get("fold"), "scaling_train_rows": split.get("scaling_train_rows"),
                    **record,
                })
            run_row = {
                "cache_key": keys[model_name], "configuration_id": CONFIGURATION_ID,
                "experiment": experiment, "dataset": dataset_name, "model": model_name,
                "implementation": getattr(model, "model_name", type(model).__name__),
                "split_id": split["split_id"], "repeat": split.get("repeat"), "fold": split.get("fold"),
                "scaling_train_rows": split.get("scaling_train_rows"),
                "train_rows": int(len(train_index)), "test_rows": int(len(test_index)),
                "n_features": int(X_array.shape[1]), "n_classes": int(len(classes)),
                "rmse_true_class_probability": rmse, "zero_one_loss": zero_one,
                "accuracy": 1.0 - zero_one, "nll": nll, "multiclass_log_loss": nll,
                "multiclass_brier_score": brier, "roc_auc": roc_auc,
                "average_precision": average_precision, "pr_auc": average_precision,
                "preprocessing_seconds": preprocessing_seconds,
                "learner_training_seconds": learner_seconds,
                "total_training_seconds": preprocessing_seconds + learner_seconds,
                "prediction_seconds": prediction_seconds,
                "total_pipeline_seconds": preprocessing_seconds + learner_seconds + prediction_seconds,
                "learner_passes": passes, "learner_rows_processed": learner_rows,
                "preprocessing_fit_rows": fit_rows,
                "learner_training_throughput_rows_per_second": learner_rows / learner_seconds if learner_seconds else np.inf,
                "prediction_throughput_rows_per_second": len(test_index) / prediction_seconds if prediction_seconds else np.inf,
                "model_state_bytes": int(model.model_state_bytes()),
                "peak_process_rss_bytes": float(np.nanmax([rss_before, rss_after])) if np.any(np.isfinite([rss_before, rss_after])) else np.nan,
                "model_parameter_count": parameter_count, "model_context_count": context_count,
                "k": configured_k, "kmax": kmax, "selected_k": selected_k,
                "selected_feature_count": selected_count,
                "selected_feature_fraction": selected_count / X_array.shape[1],
                "selected_feature_names": json.dumps(selected_names),
                "selection_rmse": getattr(model, "selection_best_rmse_", np.nan),
                "discriminative_passes": int(getattr(model, "discriminative_passes", 0)),
                "discriminative_updates": int(getattr(model, "discriminative_updates_", 0)),
                "optimizer": getattr(model, "optimizer", None),
                "learning_rate": getattr(model, "learning_rate", None),
                "pass_stage_names": json.dumps([row["stage"] for row in model_pass_rows]),
                "pass_stage_rows": json.dumps([row["rows_processed"] for row in model_pass_rows]),
                "pass_stage_seconds": json.dumps([row["seconds"] for row in model_pass_rows]),
            }
            cache.append(run_row, model_pass_rows)
            run_rows.append(run_row); pass_rows.extend(model_pass_rows)
            print(f"DONE {dataset_name} {split['split_id']} {model_name}")
            del model
            gc.collect()
    return EvaluationResult(pd.DataFrame(run_rows), pd.DataFrame(pass_rows))


### Leakage audit and complete correctness-suite execution


In [11]:
class SpyPreprocessor:
    fitted_id_sets = []

    def fit(self, X, y):
        self.fit_rows_ = len(X)
        self.__class__.fitted_id_sets.append(frozenset(X[:, 0].astype(int)))
        return self

    def transform(self, X):
        return np.asarray(X[:, 1:], dtype=np.int64)


def test_preprocessing_is_fitted_only_on_each_training_fold():
    SpyPreprocessor.fitted_id_sets = []
    y = np.repeat(np.arange(3), 8)
    X = np.column_stack([np.arange(len(y)), y, np.arange(len(y)) % 4]).astype(np.int64)
    config = EvaluationConfig(rounds=2, folds=2, random_state=11, include_binary_metrics=False)
    result = evaluate_models(
        X, y, {"NB": lambda: KDBClassifier(k=0)}, dataset_name="leakage-test",
        config=config, preprocessor_factory=SpyPreprocessor,
        metadata={"name": "leakage-test"},
        cache=ResultCache(RESULT_CACHE_ROOT, "tests", enabled=False),
        experiment="correctness-test",
    )
    splitter = RepeatedStratifiedKFold(n_splits=2, n_repeats=2, random_state=11)
    expected = [frozenset(X[train_index, 0].astype(int)) for train_index, _ in splitter.split(X, y)]
    assert SpyPreprocessor.fitted_id_sets == expected
    assert len(SpyPreprocessor.fitted_id_sets) == 4
    assert np.all(result.run_results["preprocessing_fit_rows"] == len(X) // 2)


def test_evaluator_records_complete_multiclass_metrics_and_audit():
    rng = np.random.default_rng(4)
    y = np.repeat(np.arange(3), 10)
    identifiers = np.arange(len(y))
    X = np.column_stack([identifiers, y, rng.integers(0, 3, len(y)), rng.integers(0, 4, len(y))]).astype(np.int64)
    result = evaluate_models(
        X, y, {"KDB-0": lambda: KDBClassifier(k=0)}, dataset_name="multiclass-test",
        config=EvaluationConfig(rounds=2, folds=2, random_state=3, include_binary_metrics=True),
        preprocessor_factory=SpyPreprocessor, metadata={"name": "multiclass-test"},
        cache=ResultCache(RESULT_CACHE_ROOT, "tests", enabled=False), experiment="correctness-test",
    )
    required = {
        "rmse_true_class_probability", "zero_one_loss", "accuracy", "nll",
        "multiclass_brier_score", "roc_auc", "average_precision",
        "preprocessing_seconds", "learner_training_seconds", "total_pipeline_seconds",
        "learner_passes", "learner_rows_processed", "prediction_seconds",
        "model_state_bytes", "model_parameter_count", "model_context_count",
        "selected_feature_fraction", "pass_stage_names",
    }
    assert required.issubset(result.run_results.columns)
    assert result.run_results["roc_auc"].isna().all()
    assert np.all(result.run_results["learner_passes"] == 1)
    assert len(result.pass_audit) == 4


REQUESTED_TESTS = [
    test_headline_capacity_is_matched_at_five,
    test_kdb_k_zero_is_true_one_pass_naive_bayes,
    test_all_model_probabilities_are_normalized_and_multiclass_safe,
    test_pass_counts_and_rows_are_exact,
    test_skdb_selected_k_is_bounded_by_kmax,
    test_dkdb_zero_discriminative_passes_equals_kdb,
    test_sdkdb_zero_discriminative_passes_equals_skdb,
    test_discriminative_stage_keeps_generative_theta_fixed,
    test_preprocessing_is_fitted_only_on_each_training_fold,
    test_evaluator_records_complete_multiclass_metrics_and_audit,
]

for test in REQUESTED_TESTS:
    test()
    print(f"PASS — {test.__name__}")
print(f"\nAll {len(REQUESTED_TESTS)} correctness/leakage tests passed before real-data benchmarking.")


PASS — test_headline_capacity_is_matched_at_five
PASS — test_kdb_k_zero_is_true_one_pass_naive_bayes
PASS — test_all_model_probabilities_are_normalized_and_multiclass_safe
PASS — test_pass_counts_and_rows_are_exact
PASS — test_skdb_selected_k_is_bounded_by_kmax
PASS — test_dkdb_zero_discriminative_passes_equals_kdb
PASS — test_sdkdb_zero_discriminative_passes_equals_skdb
PASS — test_discriminative_stage_keeps_generative_theta_fixed
DONE leakage-test repeat=1,fold=1 NB
DONE leakage-test repeat=1,fold=2 NB
DONE leakage-test repeat=2,fold=1 NB
DONE leakage-test repeat=2,fold=2 NB
PASS — test_preprocessing_is_fitted_only_on_each_training_fold
DONE multiclass-test repeat=1,fold=1 KDB-0
DONE multiclass-test repeat=1,fold=2 KDB-0
DONE multiclass-test repeat=2,fold=1 KDB-0
DONE multiclass-test repeat=2,fold=2 KDB-0
PASS — test_evaluator_records_complete_multiclass_metrics_and_audit

All 10 correctness/leakage tests passed before real-data benchmarking.


PASS — test_pass_counts_and_rows_are_exact
PASS — test_skdb_selected_k_is_bounded_by_kmax
PASS — test_dkdb_zero_discriminative_passes_equals_kdb


PASS — test_sdkdb_zero_discriminative_passes_equals_skdb
PASS — test_discriminative_stage_keeps_generative_theta_fixed
DONE leakage-test repeat=1,fold=1 NB


DONE leakage-test repeat=1,fold=2 NB
DONE leakage-test repeat=2,fold=1 NB
DONE leakage-test repeat=2,fold=2 NB


PASS — test_preprocessing_is_fitted_only_on_each_training_fold
DONE multiclass-test repeat=1,fold=1 KDB-0
DONE multiclass-test repeat=1,fold=2 KDB-0
DONE multiclass-test repeat=2,fold=1 KDB-0


DONE multiclass-test repeat=2,fold=2 KDB-0
PASS — test_evaluator_records_complete_multiclass_metrics_and_audit

All 10 correctness/leakage tests passed before real-data benchmarking.


## 7. Benchmark orchestration and smoke validation


In [12]:
def stress_scaling_split_plans(n_rows, test_start, sizes, seed):
    if test_start is None or n_rows <= test_start:
        raise ValueError("Full ordered stress data is required for the documented held-out split")
    train_pool = np.arange(test_start, dtype=np.int64)
    test_index = np.arange(test_start, n_rows, dtype=np.int64)
    permutation = np.random.default_rng(seed).permutation(train_pool)
    plans = []
    seen_sizes=set()
    for requested in sizes:
        actual = len(train_pool) if requested is None else min(int(requested), len(train_pool))
        # Do not run the same effective training size twice.
        if actual in seen_sizes:
            continue
        seen_sizes.add(actual)

        train_index = permutation[:actual]
        label = "full" if actual == len(train_pool) else str(actual)

        plans.append({
            "split_id": f"documented-holdout,n={label}",
            "repeat": 1,
            "fold": 1,
            "train_index": train_index,
            "test_index": test_index,
            "scaling_train_rows": actual,
        })

    return plans


def run_selected_mode():
    mode_config = ACTIVE_MODE
    evaluation_config = EvaluationConfig(
        rounds=int(mode_config["repeats"]), folds=int(mode_config["folds"]),
        random_state=RANDOM_SEED, include_binary_metrics=True,
    )
    cache = ResultCache(RESULT_CACHE_ROOT, CONFIGURATION_ID, ENABLE_RESULT_CACHE)
    run_frames, pass_frames, metadata_rows, attempts = [], [], [], []
    for dataset_name in mode_config["datasets"]:
        row_limit = mode_config["row_limits"].get(dataset_name)
        attempted = {
            "dataset": dataset_name, "mode": MODE, "requested_row_limit": row_limit,
            "configuration_id": CONFIGURATION_ID, "status": "started", "reason": None,
        }
        try:
            X, y, metadata = load_dataset(dataset_name, max_rows=row_limit, seed=RANDOM_SEED)
            effective_kmax = min(KMAX, X.shape[1] - 1)
            metadata_rows.append(metadata_display_row(metadata))
            display(pd.DataFrame([metadata_display_row(metadata)]))
            if mode_config["use_predefined_holdout"]:
                split_plan = stress_scaling_split_plans(
                    len(X), metadata.get("predefined_test_start"), STRESS_SCALING_ROWS, RANDOM_SEED,
                )
            else:
                split_plan = None
            result = evaluate_models(
                X, y, headline_model_factories(kmax=effective_kmax), dataset_name=dataset_name,
                config=evaluation_config, split_plan=split_plan, metadata=metadata,
                cache=cache, experiment="headline",
            )
            run_frames.append(result.run_results); pass_frames.append(result.pass_audit)
            attempted["status"] = "complete"
        except (KeyboardInterrupt, MemoryError) as exc:
            attempted.update(status="incomplete", reason=f"{type(exc).__name__}: {exc}")
            print(f"INCOMPLETE {dataset_name}: {attempted['reason']}")
            if MODE != "STRESS":
                raise
        except Exception as exc:
            attempted.update(status="failed", reason=f"{type(exc).__name__}: {exc}")
            print(f"FAILED {dataset_name}: {attempted['reason']}")
            if MODE != "STRESS":
                raise
        finally:
            attempts.append(attempted)
    runs = pd.concat(run_frames, ignore_index=True) if run_frames else pd.DataFrame()
    passes = pd.concat(pass_frames, ignore_index=True) if pass_frames else pd.DataFrame()
    return runs, passes, pd.DataFrame(metadata_rows), pd.DataFrame(attempts)

## 8. Core experiments

Set `MODE="CORE"` in the single configuration cell and restart/run all.
CORE uses full Skin, MiniBooNE, cod-rna, Covertype, and
YearPredictionMSD with one shared set of 3-fold stratified indices and
one repeat. A failure is surfaced rather than relabelled as a full run.


In [13]:
assert MODE == "CORE"
assert set(ACTIVE_MODE["datasets"]) == {
    "Skin Segmentation",
    "MiniBooNE",
    "cod-rna",
    "Covertype",
    "YearPredictionMSD",
}
assert ACTIVE_MODE["folds"] == 3
assert ACTIVE_MODE["repeats"] == 1

print("MODE:", MODE)
print("DATASETS:", ACTIVE_MODE["datasets"])
print("FOLDS:", ACTIVE_MODE["folds"])
print("REPEATS:", ACTIVE_MODE["repeats"])
print("ROW LIMITS:", ACTIVE_MODE["row_limits"])
print("CONFIGURATION ID:", CONFIGURATION_ID)

MODE: CORE
DATASETS: ('Skin Segmentation', 'MiniBooNE', 'cod-rna', 'Covertype', 'YearPredictionMSD')
FOLDS: 3
REPEATS: 1
ROW LIMITS: {}
CONFIGURATION ID: 53e7c82e27eb953c


In [ ]:
RUNS = pd.DataFrame()
PASSES = pd.DataFrame()
DATASET_METADATA = pd.DataFrame()
ATTEMPT_STATUS = pd.DataFrame()
if RUN_BENCHMARKS:
    RUNS, PASSES, DATASET_METADATA, ATTEMPT_STATUS = run_selected_mode()
    display(ATTEMPT_STATUS)
    print(f"Completed run rows: {len(RUNS)} | pass rows: {len(PASSES)}")
else:
    print("Benchmarks disabled; definitions and correctness tests still executed.")


,name,source,url,n_rows,n_features,n_classes,class_counts,class_proportions,missing_feature_values,nonfinite_feature_values,numeric_dtype_compatible,feature_dtype_summary,approx_raw_memory_bytes,sampled,requested_row_limit
0,Skin Segmentation,UCI Skin Segmentation via OpenML data_id=1502,https://www.openml.org/d/1502,245057,3,2,"{""1"": 50859, ""2"": 194198}","{""1"": 0.20753947040892526, ""2"": 0.792460529591...",0,0,True,"{""int64"": 3}",6126905,False,None


RESUME Skin Segmentation repeat=1,fold=1: all models loaded
RESUME Skin Segmentation repeat=1,fold=2: all models loaded
RESUME Skin Segmentation repeat=1,fold=3: all models loaded


,name,source,url,n_rows,n_features,n_classes,class_counts,class_proportions,missing_feature_values,nonfinite_feature_values,numeric_dtype_compatible,feature_dtype_summary,approx_raw_memory_bytes,sampled,requested_row_limit
0,MiniBooNE,UCI MiniBooNE via OpenML data_id=41150,https://www.openml.org/d/41150,130064,50,2,"{""True"": 36499, ""False"": 93565}","{""True"": 0.28062338541025955, ""False"": 0.71937...",0,0,True,"{""float64"": 50}",52156151,False,None


RESUME MiniBooNE repeat=1,fold=1: all models loaded
RESUME MiniBooNE repeat=1,fold=2: all models loaded
RESUME MiniBooNE repeat=1,fold=3: all models loaded


,name,source,url,n_rows,n_features,n_classes,class_counts,class_proportions,missing_feature_values,nonfinite_feature_values,numeric_dtype_compatible,feature_dtype_summary,approx_raw_memory_bytes,sampled,requested_row_limit
0,cod-rna,LIBSVM cod-rna train/validation/test files,https://www.csie.ntu.edu.tw/~cjlin/libsvmtools...,488565,8,2,"{""-1"": 325710, ""1"": 162855}","{""-1"": 0.6666666666666666, ""1"": 0.333333333333...",0,0,True,"{""float64"": 8}",31756981,False,None


DONE cod-rna repeat=1,fold=1 sKDB
DONE cod-rna repeat=1,fold=1 dKDB
DONE cod-rna repeat=1,fold=1 sdKDB (experimental/unpublished)
DONE cod-rna repeat=1,fold=2 KDB
DONE cod-rna repeat=1,fold=2 sKDB
DONE cod-rna repeat=1,fold=2 dKDB
DONE cod-rna repeat=1,fold=2 sdKDB (experimental/unpublished)
DONE cod-rna repeat=1,fold=3 KDB
DONE cod-rna repeat=1,fold=3 sKDB
DONE cod-rna repeat=1,fold=3 dKDB
DONE cod-rna repeat=1,fold=3 sdKDB (experimental/unpublished)


,name,source,url,n_rows,n_features,n_classes,class_counts,class_proportions,missing_feature_values,nonfinite_feature_values,numeric_dtype_compatible,feature_dtype_summary,approx_raw_memory_bytes,sampled,requested_row_limit
0,Covertype,UCI Covertype via sklearn fetch_covtype,https://archive.ics.uci.edu/dataset/31/covertype,581012,54,7,"{""1"": 211840, ""2"": 283301, ""3"": 35754, ""4"": 27...","{""1"": 0.36460520608868663, ""2"": 0.487599223423...",0,0,True,"{""float64"": 54}",253321488,False,None


DONE Covertype repeat=1,fold=1 KDB


In [ ]:
OUTPUT_DIR = Path("benchmark_results") / CONFIGURATION_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUNS.to_csv(OUTPUT_DIR / "runs.csv", index=False)
PASSES.to_csv(OUTPUT_DIR / "passes.csv", index=False)
DATASET_METADATA.to_csv(OUTPUT_DIR / "dataset_metadata.csv", index=False)
ATTEMPT_STATUS.to_csv(OUTPUT_DIR / "attempt_status.csv", index=False)

print("Saved final outputs to:", OUTPUT_DIR.resolve())

## 9. Stress and nested scaling experiments

Set `MODE="STRESS"` and restart/run all. SUSY uses rows 0–4,499,999
for the nested training pool and the final 500k for test; HIGGS uses rows
0–10,499,999 and the final 500k for test. One seeded permutation creates
nested prefixes at 100k, 250k, 500k, 1m, 2m, and full training size.
Incomplete configurations remain explicitly recorded in `ATTEMPT_STATUS`.


In [ ]:
if MODE == "STRESS":
    assert ACTIVE_MODE["use_predefined_holdout"]
    print("STRESS documented holdouts and nested scaling prefixes are active.")
else:
    print("STRESS not active; no large download was performed.")


### Separate `k=0…5` sensitivity experiment

Enable `RUN_K_SENSITIVITY` only when the extra learner passes are desired.
Its cache and rows are labelled `experiment="k_sensitivity"`; it never
replaces the matched four-model headline result.


In [ ]:
SENSITIVITY_RUNS = pd.DataFrame()
if RUN_K_SENSITIVITY:
    sensitivity_frames = []
    sensitivity_config = EvaluationConfig(
        rounds=int(ACTIVE_MODE["repeats"]), folds=int(ACTIVE_MODE["folds"]),
        random_state=RANDOM_SEED, include_binary_metrics=True,
    )
    sensitivity_cache = ResultCache(RESULT_CACHE_ROOT, CONFIGURATION_ID, ENABLE_RESULT_CACHE)
    for dataset_name in ACTIVE_MODE["datasets"]:
        if MODE == "STRESS":
            print(f"Sensitivity skipped for {dataset_name} STRESS; run an explicitly budgeted subset instead.")
            continue
        X, y, metadata = load_dataset(
            dataset_name, max_rows=ACTIVE_MODE["row_limits"].get(dataset_name), seed=RANDOM_SEED,
        )
        result = evaluate_models(
            X, y, sensitivity_model_factories(), dataset_name=dataset_name,
            config=sensitivity_config, metadata=metadata, cache=sensitivity_cache,
            experiment="k_sensitivity",
        )
        sensitivity_frames.append(result.run_results)
    if sensitivity_frames:
        SENSITIVITY_RUNS = pd.concat(sensitivity_frames, ignore_index=True)
        display(SENSITIVITY_RUNS.groupby(["dataset", "model", "k"])[["rmse_true_class_probability", "zero_one_loss"]].mean())
else:
    print("Separate k sensitivity is disabled.")


## 10. Statistical analysis and authoritative derived tables


In [ ]:
MODEL_ORDER = ["KDB", "sKDB", "dKDB", "sdKDB (experimental/unpublished)"]
PAIRWISE_COMPARISONS = [
    ("sKDB", "KDB"),
    ("dKDB", "KDB"),
    ("sdKDB (experimental/unpublished)", "sKDB"),
    ("sdKDB (experimental/unpublished)", "dKDB"),
]
COMPARISON_METRICS = [
    "rmse_true_class_probability", "zero_one_loss", "nll",
    "learner_training_seconds", "model_state_bytes",
]


def per_dataset_model_summary(runs):
    if runs.empty:
        return pd.DataFrame()
    metrics = [
        "rmse_true_class_probability", "zero_one_loss", "accuracy", "nll",
        "multiclass_brier_score", "roc_auc", "average_precision",
        "preprocessing_seconds", "learner_training_seconds", "total_pipeline_seconds",
        "learner_passes", "learner_rows_processed", "prediction_seconds",
        "learner_training_throughput_rows_per_second",
        "prediction_throughput_rows_per_second", "model_state_bytes",
        "model_parameter_count", "model_context_count", "selected_k",
        "selected_feature_count", "selected_feature_fraction", "selection_rmse",
        "discriminative_passes", "discriminative_updates",
    ]
    usable = [metric for metric in metrics if metric in runs]
    grouped = runs.groupby(["dataset", "model"], sort=False)[usable].agg(["mean", "std", "count"])
    grouped.columns = [f"{metric}_{stat}" for metric, stat in grouped.columns]
    return grouped.reset_index()


def mean_std_table(runs):
    summary = per_dataset_model_summary(runs)
    if summary.empty:
        return summary
    output = summary[["dataset", "model"]].copy()
    for metric in ["rmse_true_class_probability", "zero_one_loss", "nll", "learner_training_seconds", "model_state_bytes"]:
        output[metric] = summary.apply(
            lambda row: f"{row[f'{metric}_mean']:.6g} ± {row[f'{metric}_std']:.3g}" if pd.notna(row[f'{metric}_std']) else f"{row[f'{metric}_mean']:.6g} ± n/a",
            axis=1,
        )
    return output


def comparison_dataset_means(runs):
    if runs.empty:
        return pd.DataFrame()
    selected = runs.copy()
    if "scaling_train_rows" in selected and selected["scaling_train_rows"].notna().any():
        maxima = selected.groupby("dataset")["scaling_train_rows"].transform("max")
        selected = selected[selected["scaling_train_rows"].isna() | (selected["scaling_train_rows"] == maxima)]
    return selected.groupby(["dataset", "model"], as_index=False)[COMPARISON_METRICS].mean()


def model_rank_table(runs, metric="rmse_true_class_probability"):
    means = comparison_dataset_means(runs)
    if means.empty:
        return pd.DataFrame()
    pivot = means.pivot(index="dataset", columns="model", values=metric).reindex(columns=MODEL_ORDER)
    complete = pivot.dropna()
    if complete.empty:
        return pd.DataFrame()
    ranks = complete.apply(lambda row: pd.Series(rankdata(row, method="average"), index=row.index), axis=1)
    ranks.index.name = "dataset"
    return ranks.reset_index()


def pairwise_delta_table(runs, alpha=0.05):
    means = comparison_dataset_means(runs)
    records = []
    for left, right in PAIRWISE_COMPARISONS:
        for metric in COMPARISON_METRICS:
            pivot = means.pivot(index="dataset", columns="model", values=metric)
            if left not in pivot or right not in pivot:
                continue
            deltas = (pivot[left] - pivot[right]).dropna().to_numpy(dtype=float)
            n = len(deltas)
            mean_delta = float(np.mean(deltas)) if n else np.nan
            sd = float(np.std(deltas, ddof=1)) if n > 1 else np.nan
            margin = float(t.ppf(1 - alpha / 2, n - 1) * sd / np.sqrt(n)) if n > 1 else np.nan
            records.append({
                "comparison": f"{left} − {right}", "metric": metric, "n_datasets": n,
                "mean_delta": mean_delta, "std_delta": sd,
                "ci95_low": mean_delta - margin if n > 1 else np.nan,
                "ci95_high": mean_delta + margin if n > 1 else np.nan,
                "wins": int(np.sum(deltas < -1e-12)), "draws": int(np.sum(np.abs(deltas) <= 1e-12)),
                "losses": int(np.sum(deltas > 1e-12)),
                "direction": "negative favors first-named model (all listed metrics are lower-is-better)",
            })
    return pd.DataFrame(records)


def win_draw_loss_table(delta_table):
    columns = ["comparison", "metric", "n_datasets", "wins", "draws", "losses"]
    return delta_table[columns].copy() if not delta_table.empty else pd.DataFrame(columns=columns)


def friedman_nemenyi(runs, alpha=0.05):
    means = comparison_dataset_means(runs)
    if means.empty:
        return {"status": "insufficient data"}, pd.DataFrame()
    pivot = means.pivot(index="dataset", columns="model", values="rmse_true_class_probability").reindex(columns=MODEL_ORDER).dropna()
    if len(pivot) < 3:
        return {"status": "requires at least 3 complete datasets", "n_datasets": len(pivot)}, pd.DataFrame()
    statistic, p_value = friedmanchisquare(*[pivot[column].to_numpy() for column in MODEL_ORDER])
    ranks = pivot.apply(lambda row: rankdata(row, method="average"), axis=1, result_type="expand")
    ranks.columns = MODEL_ORDER
    average_ranks = ranks.mean(axis=0)
    result = {
        "status": "complete", "n_datasets": len(pivot), "n_models": len(MODEL_ORDER),
        "friedman_statistic": float(statistic), "friedman_p_value": float(p_value),
        "alpha": alpha, "average_ranks": average_ranks.to_dict(),
        "folds_treated_as_datasets": False,
    }
    records = []
    if p_value < alpha:
        k_models, n_datasets = len(MODEL_ORDER), len(pivot)
        standard_error = math.sqrt(k_models * (k_models + 1) / (6 * n_datasets))
        critical_q = float(studentized_range.ppf(1 - alpha, k_models, np.inf) / math.sqrt(2))
        result["nemenyi_critical_difference"] = critical_q * standard_error
        for i, left in enumerate(MODEL_ORDER):
            for right in MODEL_ORDER[i + 1:]:
                difference = abs(average_ranks[left] - average_ranks[right])
                p_posthoc = float(studentized_range.sf((difference / standard_error) * math.sqrt(2), k_models, np.inf))
                records.append({
                    "model_a": left, "model_b": right, "rank_difference": difference,
                    "p_value": p_posthoc, "significant": p_posthoc < alpha,
                })
    else:
        result["nemenyi_critical_difference"] = None
    return result, pd.DataFrame(records)


def supplementary_wilcoxon_holm(runs):
    means = comparison_dataset_means(runs)
    tests = []
    for left, right in PAIRWISE_COMPARISONS:
        pivot = means.pivot(index="dataset", columns="model", values="rmse_true_class_probability")
        if left not in pivot or right not in pivot:
            continue
        paired = pivot[[left, right]].dropna()
        if len(paired) < 3 or np.allclose(paired[left], paired[right]):
            continue
        statistic, p_value = wilcoxon(paired[left], paired[right])
        tests.append({"comparison": f"{left} − {right}", "statistic": statistic, "p_value": p_value})
    tests.sort(key=lambda row: row["p_value"])
    m = len(tests)
    running = 0.0
    for rank, row in enumerate(tests):
        adjusted = min(1.0, (m - rank) * row["p_value"])
        running = max(running, adjusted)
        row["holm_adjusted_p"] = running
    return pd.DataFrame(tests)


DATASET_MODEL_SUMMARY = per_dataset_model_summary(RUNS)
MEAN_STD_TABLE = mean_std_table(RUNS)
COMPUTATIONAL_EFFICIENCY_TABLE = DATASET_MODEL_SUMMARY[[column for column in DATASET_MODEL_SUMMARY.columns if column in {"dataset", "model"} or any(token in column for token in ("seconds", "throughput", "passes", "rows_processed", "state_bytes", "parameter_count", "context_count"))]] if not DATASET_MODEL_SUMMARY.empty else pd.DataFrame()
MODEL_SELECTION_TABLE = RUNS[[column for column in ["dataset", "split_id", "model", "k", "kmax", "selected_k", "selected_feature_count", "selected_feature_fraction", "selected_feature_names", "selection_rmse", "discriminative_passes", "discriminative_updates", "optimizer", "learning_rate"] if column in RUNS]].copy() if not RUNS.empty else pd.DataFrame()
DATASET_LEVEL_RANKS = model_rank_table(RUNS)
PAIRWISE_EFFECT_TABLE = pairwise_delta_table(RUNS)
WIN_DRAW_LOSS_TABLE = win_draw_loss_table(PAIRWISE_EFFECT_TABLE)
FRIEDMAN_RESULT, NEMENYI_TABLE = friedman_nemenyi(RUNS)
WILCOXON_HOLM_TABLE = supplementary_wilcoxon_holm(RUNS)

print("Friedman analysis (dataset-level means; CV folds are not datasets):")
display(pd.DataFrame([FRIEDMAN_RESULT]))
if not NEMENYI_TABLE.empty:
    display(NEMENYI_TABLE)


## 11. Final figures


In [ ]:
def _no_data(axis, message):
    axis.text(0.5, 0.5, message, ha="center", va="center", transform=axis.transAxes)
    axis.set_xticks([]); axis.set_yticks([])


def plot_pairwise_rmse(runs):
    means = comparison_dataset_means(runs)
    figure, axes = plt.subplots(2, 2, figsize=(12, 10))
    for axis, (left, right) in zip(axes.ravel(), PAIRWISE_COMPARISONS):
        pivot = means.pivot(index="dataset", columns="model", values="rmse_true_class_probability") if not means.empty else pd.DataFrame()
        if left not in pivot or right not in pivot:
            _no_data(axis, "No complete comparison")
            continue
        paired = pivot[[right, left]].dropna()
        if paired.empty:
            _no_data(axis, "No complete comparison")
            continue
        low = float(paired.min().min()); high = float(paired.max().max())
        pad = max((high - low) * 0.08, 1e-4)
        axis.plot([low - pad, high + pad], [low - pad, high + pad], "k--", linewidth=1)
        axis.scatter(paired[right], paired[left], s=60)
        for name, row in paired.iterrows():
            axis.annotate(name, (row[right], row[left]), xytext=(4, 4), textcoords="offset points", fontsize=8)
        axis.set(xlabel=f"{right} RMSE", ylabel=f"{left} RMSE", title=f"{right} vs {left}")
    figure.suptitle("JMLR-style dataset-level pairwise true-class RMSE")
    figure.tight_layout(); return figure


def plot_pass_tradeoffs(runs):
    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    for axis, metric, label in zip(axes, ["rmse_true_class_probability", "nll"], ["RMSE", "NLL"]):
        if runs.empty:
            _no_data(axis, "No run data"); continue
        for model, group in runs.groupby("model", sort=False):
            axis.scatter(group["learner_passes"], group[metric], label=model, alpha=0.75)
        axis.set(xlabel="Exact learner passes", ylabel=label, title=f"Learner passes vs {label}")
    if not runs.empty:
        axes[0].legend(fontsize=8)
    figure.tight_layout(); return figure


def plot_training_breakdown(runs, passes):
    figure, axis = plt.subplots(figsize=(12, 6))
    if runs.empty or passes.empty:
        _no_data(axis, "No timing data"); return figure
    prep = runs.groupby("model")["preprocessing_seconds"].mean().reindex(MODEL_ORDER).fillna(0)
    stages = passes.groupby(["model", "stage"])["seconds"].mean().unstack(fill_value=0).reindex(MODEL_ORDER).fillna(0)
    x = np.arange(len(MODEL_ORDER)); bottom = np.zeros(len(MODEL_ORDER))
    axis.bar(x, prep, label="preprocessing (not a learner pass)", color="#bbbbbb")
    bottom += prep.to_numpy()
    for stage in stages.columns:
        values = stages[stage].to_numpy()
        axis.bar(x, values, bottom=bottom, label=stage)
        bottom += values
    axis.set_xticks(x, MODEL_ORDER, rotation=20, ha="right")
    axis.set_ylabel("Mean seconds per split")
    axis.set_title("Training-time breakdown: preprocessing visually separated")
    axis.legend(fontsize=8, ncol=2)
    figure.tight_layout(); return figure


def plot_size_throughput_selection(runs):
    figure, axes = plt.subplots(2, 2, figsize=(13, 10))
    if runs.empty:
        for axis in axes.ravel(): _no_data(axis, "No run data")
        return figure
    for model, group in runs.groupby("model", sort=False):
        axes[0, 0].scatter(group["model_state_bytes"], group["rmse_true_class_probability"], label=model, alpha=0.75)
    axes[0, 0].set(xscale="log", xlabel="Model-state bytes", ylabel="RMSE", title="Model size vs predictive loss")
    throughput = runs.groupby("model")["prediction_throughput_rows_per_second"].mean().reindex(MODEL_ORDER)
    axes[0, 1].bar(np.arange(len(throughput)), throughput)
    axes[0, 1].set_xticks(np.arange(len(throughput)), throughput.index, rotation=20, ha="right")
    axes[0, 1].set(ylabel="Rows / second", title="Prediction throughput")
    selective = runs[runs["model"].isin(["sKDB", "sdKDB (experimental/unpublished)"])]
    if selective.empty:
        _no_data(axes[1, 0], "No selective runs"); _no_data(axes[1, 1], "No selective runs")
    else:
        data = [selective.loc[selective["model"] == model, "selected_feature_fraction"].to_numpy() for model in ["sKDB", "sdKDB (experimental/unpublished)"]]
        axes[1, 0].boxplot(data, tick_labels=["sKDB", "sdKDB (experimental/unpublished)"], showmeans=True)
        axes[1, 0].set(ylabel="Selected feature fraction", title="Selective feature fraction")
        for model, group in selective.groupby("model", sort=False):
            axes[1, 1].hist(group["selected_k"], bins=np.arange(-0.5, KMAX + 1.5), alpha=0.55, label=model)
        axes[1, 1].set(xlabel="Selected k", ylabel="Split count", title="Selected-k distribution")
        axes[1, 1].set_xticks(range(KMAX + 1)); axes[1, 1].legend(fontsize=8)
    axes[0, 0].legend(fontsize=8)
    figure.tight_layout(); return figure


def plot_average_ranks(runs, friedman_result):
    figure, axis = plt.subplots(figsize=(11, 3.2))
    ranks = model_rank_table(runs)
    if ranks.empty:
        _no_data(axis, "Insufficient complete datasets for ranks"); return figure
    average = ranks[MODEL_ORDER].mean().sort_values()
    y = np.zeros(len(average))
    axis.scatter(average.values, y, s=90)
    for index, (model, rank) in enumerate(average.items()):
        axis.annotate(model, (rank, 0), xytext=(0, 12 + 14 * (index % 2)), textcoords="offset points", ha="center", fontsize=8)
    axis.set_xlim(0.75, len(MODEL_ORDER) + 0.25); axis.set_ylim(-0.2, 0.55)
    axis.set_yticks([]); axis.set_xlabel("Average rank (lower is better)")
    title = "Dataset-level average ranks"
    cd = friedman_result.get("nemenyi_critical_difference")
    if cd is not None:
        start = 1.0; axis.plot([start, start + cd], [0.38, 0.38], color="black", linewidth=3)
        axis.text(start + cd / 2, 0.42, f"Nemenyi CD={cd:.3f}", ha="center", fontsize=8)
        title += " with significant-omnibus Nemenyi CD"
    else:
        title += " (no CD: omnibus not significant or insufficient datasets)"
    axis.set_title(title); figure.tight_layout(); return figure


def plot_stress_scaling(runs):
    figure, axes = plt.subplots(2, 2, figsize=(13, 10))
    stress = runs[runs["dataset"].isin(["SUSY", "HIGGS"]) & runs["scaling_train_rows"].notna()] if not runs.empty else pd.DataFrame()
    specs = [
        ("learner_training_seconds", "Learner training seconds"),
        ("rmse_true_class_probability", "RMSE"),
        ("selected_k", "Selected k"),
        ("selected_feature_fraction", "Selected feature fraction"),
    ]
    if stress.empty:
        for axis in axes.ravel(): _no_data(axis, "Run STRESS mode to populate")
        return figure
    aggregate = stress.groupby(["dataset", "model", "scaling_train_rows"], as_index=False)[[item[0] for item in specs]].mean()
    for axis, (metric, label) in zip(axes.ravel(), specs):
        for (dataset, model), group in aggregate.groupby(["dataset", "model"], sort=False):
            axis.plot(group["scaling_train_rows"], group[metric], marker="o", label=f"{dataset}: {model}")
        axis.set(xlabel="Training rows", ylabel=label, title=f"Stress scaling: {label}")
        axis.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
    axes[0, 0].legend(fontsize=7, ncol=2)
    figure.tight_layout(); return figure


FINAL_FIGURES = [
    plot_pairwise_rmse(RUNS),
    plot_pass_tradeoffs(RUNS),
    plot_training_breakdown(RUNS, PASSES),
    plot_size_throughput_selection(RUNS),
    plot_average_ranks(RUNS, FRIEDMAN_RESULT),
    plot_stress_scaling(RUNS),
]
plt.show()
print(f"All {len(FINAL_FIGURES)} final figure functions executed.")


## 12. Final comparison tables and concise findings


In [ ]:
AUTHORITATIVE_TABLES = {
    "dataset_metadata": DATASET_METADATA,
    "per_dataset_model_summary": DATASET_MODEL_SUMMARY,
    "mean_plus_minus_std": MEAN_STD_TABLE,
    "pass_audit": PASSES,
    "computational_efficiency": COMPUTATIONAL_EFFICIENCY_TABLE,
    "model_selection": MODEL_SELECTION_TABLE,
    "dataset_level_model_ranks": DATASET_LEVEL_RANKS,
    "win_draw_loss": WIN_DRAW_LOSS_TABLE,
    "pairwise_effects_with_95ci": PAIRWISE_EFFECT_TABLE,
    "nemenyi_posthoc": NEMENYI_TABLE,
    "wilcoxon_holm_supplement": WILCOXON_HOLM_TABLE,
    "attempt_status": ATTEMPT_STATUS,
}
for table_name, table in AUTHORITATIVE_TABLES.items():
    print(f"\n--- {table_name} ({len(table)} rows) ---")
    display(table)

if RUNS.empty:
    print("No empirical finding is claimed because no benchmark rows are available.")
else:
    best = comparison_dataset_means(RUNS)
    if not best.empty:
        winners = best.loc[best.groupby("dataset")["rmse_true_class_probability"].idxmin(), ["dataset", "model", "rmse_true_class_probability"]]
        print("Concise descriptive finding (not a significance claim): lowest mean RMSE by dataset")
        display(winners)

expected_columns = {
    "dataset", "split_id", "model", "rmse_true_class_probability", "zero_one_loss",
    "accuracy", "nll", "multiclass_brier_score", "learner_passes",
    "learner_rows_processed", "preprocessing_seconds", "learner_training_seconds",
    "prediction_seconds", "total_pipeline_seconds", "model_state_bytes",
    "selected_k", "selected_feature_fraction", "discriminative_passes",
}
if RUN_BENCHMARKS:
    assert not RUNS.empty
    assert expected_columns.issubset(RUNS.columns)
    assert set(RUNS["model"]) == set(MODEL_ORDER)
    assert RUNS.groupby(["dataset", "split_id"])["model"].nunique().eq(4).all()
    assert (RUNS["configuration_id"] == CONFIGURATION_ID).all()
    print("FINAL VALIDATION PASS — tidy schema, four learners per split, and configuration IDs are consistent.")


### Interpretation guardrails

- Smoke results validate execution and accounting only; they are not
  research conclusions.
- Friedman/Nemenyi uses dataset-level model means. CV folds are never
  treated as independent datasets.
- Negative deltas in the pairwise table favor the first-named model
  because every requested comparison metric is lower-is-better.
- Peak RSS is `NaN` on platforms without a robust process high-water RSS
  API; no sampled snapshot is mislabeled as a peak.
- Full stress failures or interruptions are recorded, never silently
  relabelled as full-data results.
- sdKDB remains **experimental/unpublished** throughout.
